# Question 1 — Group project: us and the AI
### Course: 21046 Data Science and Machine Learning for Finance (Bocconi University, 2025/2026)
## Premises on the use of AI

Most of the code below was directly generated by AI. We told it what we needed and then inspected the code that was given to us. In case of doubt, we asked another AI to explain what the lines of code we were unsure about were doing (without telling the new AI what we expected the code to do), in order to check whether the code was actually doing what we needed.

Any comment of the finding has been written first by discussing the results among ourselves, then providing AI with our findings and checking whether it converged to the same explanation we had in mind, without explicitly stating our interpretation. In case of disagreement, we checked online and consulted other AIs to see who was right.



In [ ]:
# ============================================================
# INSTALL (run once, then restart kernel)
# ============================================================
!pip install tensorflow

# ============================================================
# STANDARD LIBRARIES
# ============================================================
import os
from datetime import datetime, timedelta

# ============================================================
# DATA / NUMERICAL
# ============================================================
import numpy as np
import pandas as pd

# ============================================================
# VISUALIZATION
# ============================================================
import matplotlib.pyplot as plt
import plotly.express as px

# ============================================================
# DATA SOURCES
# ============================================================
import yfinance as yf

# ============================================================
# SCIENTIFIC / STATS
# ============================================================
from scipy import stats
from scipy.optimize import differential_evolution, minimize
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

from statsmodels.stats.diagnostic import lilliefors

# ============================================================
# MACHINE LEARNING
# ============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.metrics import davies_bouldin_score

# ============================================================
# OPTIMIZATION
# ============================================================
import cvxpy as cp

# ============================================================
# TENSORFLOW / AUTOENCODER
# ============================================================
import tensorflow as tf
import tensorflow_probability as tfp

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# ============================================================
# 1) download (any public) data on stock prices (at least 100 stocks)
# ============================================================

# File names
constituents_file = "sp500_constituents.csv"
prices_file = "sp500_adj_close.csv"
market_caps_file = "sp500_market_caps.csv"

# GitHub raw base URL
github_base_url = "https://raw.githubusercontent.com/stfgrz/21046-mlf-ps/main/data/"

constituents_github_url = github_base_url + constituents_file
prices_github_url = github_base_url + prices_file
market_caps_github_url = github_base_url + market_caps_file

# Load S&P 500 constituents from GitHub; if unavailable, download from public source
try:
    sp500 = pd.read_csv(constituents_github_url)
    print("Loaded S&P 500 constituents from GitHub")

except Exception:
    url = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
    sp500 = pd.read_csv(url)
    sp500.to_csv(constituents_file, index=False)
    print("Downloaded S&P 500 constituents from public source")

# Define tickers
tickers = sp500["Symbol"].str.replace(".", "-", regex=False).tolist()

# Load adjusted close prices from GitHub; if unavailable, download from Yahoo Finance
try:
    prices_clean = pd.read_csv(
        prices_github_url,
        index_col=0,
        parse_dates=True
    )
    print("Loaded adjusted prices from GitHub")

except Exception:
    start_date = "2015-01-01"
    end_date = "2026-01-01"

    prices = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        auto_adjust=False,
        progress=False
    )["Adj Close"]

    prices_clean = prices.dropna(axis=1)
    prices_clean.to_csv(prices_file)

    print("Downloaded adjusted prices from Yahoo Finance")
    print(f"Original number of stocks: {prices.shape[1]}")
    print(f"Final number of stocks with full history: {prices_clean.shape[1]}")

# Load market caps from GitHub; if unavailable, download from Yahoo Finance
try:
    market_caps = pd.read_csv(
        market_caps_github_url,
        index_col=0
    ).squeeze("columns")
    print("Loaded market caps from GitHub")

except Exception:
    print("Downloading market caps from Yahoo Finance...")

    market_caps_dict = {}

    for ticker in prices_clean.columns:
        try:
            info = yf.Ticker(ticker).info
            market_caps_dict[ticker] = info.get("marketCap", None)
        except Exception:
            market_caps_dict[ticker] = None

    market_caps = pd.Series(market_caps_dict).dropna()
    market_caps.name = "market_cap"
    market_caps.to_csv(market_caps_file)

    print("Downloaded market caps from Yahoo Finance")
    print(f"Market cap available for {len(market_caps)} stocks")

# Keep only stocks that have both prices and market cap
common_tickers = prices_clean.columns.intersection(market_caps.index)

prices_clean = prices_clean[common_tickers]
market_caps = market_caps.loc[common_tickers]

assert prices_clean.shape[1] >= 100, "The dataset contains fewer than 100 stocks."

print("\nFinal aligned dataset:")
print(f"Number of stocks with prices: {prices_clean.shape[1]}")
print(f"Number of stocks with market cap: {len(market_caps)}")
print(f"Date range: {prices_clean.index.min().date()} to {prices_clean.index.max().date()}")

We asked AI for an easy way to download S&P 500 stock prices. The AI initially suggested downloading stock price data with yfinance. However, yfinance requires a list of ticker symbols as input, so we needed an external source for the S&P 500 constituent list. The AI suggested scraping the ticker list from Wikipedia. This turned out to be problematic.

Then we asked for a better alternative. The AI suggested using a public CSV file from the Open Knowledge Foundation on GitHub (https://github.com/datasets/s-and-p-500-companies). We checked the README file available on the GitHub repository and the website of the foundation (which we did not know before). It seems to be a reliable source.

After downloading the data, stocks with missing values across the full 10-year period were dropped, resulting in a final sample of 461 out of 500 companies. While the number of excluded companies may seem small, this introduces a non-trivial form of survivorship bias. The excluded companies are not missing at random; they are systematically the worst-performing or most troubled ones, making the remaining sample look artificially healthier than the true market.

We asked the AI whether a survivorship-bias-free analysis was possible. The historically accurate day-by-day index composition would be too complex, so we chose to focus on
 the companies currently listed in the index.

We also downloaded the current capitalization of the stocks. We will use this information in descriptive statistics to obtain weighted averages of returns. We are aware that these weights are misleading, since for the entire period we are using today’s capitalization instead of the capitalization specific to each period.

N.B. We are aware that in file A03c weekly stock prices were used to avoid microstructure noise, but we preferred to use daily data for two reasons:
1. It gives us many more observations to estimate our optimal portfolio more reliably;
2. since part of this analysis overlaps with what was done in class, working at a different frequency gave us a chance to spot whether and how the results change.

In [ ]:
# ============================================================
# 2) compute linear returns
# ============================================================

# Get and print daily returns per company
returns = prices_clean.pct_change().dropna()

print(returns.head(3))

In [ ]:

# ============================================================
# 3) compute the main descriptive statistics
# ============================================================

# Obtain and print basic statistics
print("Computing descriptive statistics for all stocks...")

mean        = returns.mean()
median      = returns.median()
std         = returns.std()
variance    = returns.var()
mad         = returns.apply(lambda x: (x - x.median()).abs().mean())
skewness    = returns.skew()
kurtosis    = returns.kurt()
min_ret     = returns.min()
max_ret     = returns.max()
q25         = returns.quantile(0.25)
q75         = returns.quantile(0.75)

desc_stats = pd.DataFrame({
    'Mean'     : mean,
    'Median'   : median,
    'Std Dev'  : std,
    'Variance' : variance,
    'MAD'      : mad,
    'Skewness' : skewness,
    'Kurtosis' : kurtosis,
    'Min'      : min_ret,
    'Max'      : max_ret,
    'Q25'      : q25,
    'Q75'      : q75
})

print("\nDescriptive Statistics for all stocks:")
display(desc_stats)


# Merging data on returns with that on market capitalization
common_tickers     = [t for t in desc_stats.index if t in market_caps.index]
desc_stats_aligned = desc_stats.loc[common_tickers]
weights            = market_caps[common_tickers]
weights            = weights / weights.sum()


# Simple and weighted average return
simple_avg      = desc_stats_aligned.mean()
simple_avg.name = 'Equal Weighted Market'

weighted_avg      = desc_stats_aligned.multiply(weights, axis=0).sum()
weighted_avg.name = 'Cap Weighted Market'

summary = pd.concat([simple_avg, weighted_avg], axis=1).T

print("\n--- Per Stock Descriptive Statistics ---")
display(desc_stats)

print("\n--- Market Level Summary ---")
display(summary)


# PLOT 1: Average daily returns
# ============================================================

avg_returns = returns.mean(axis=1)

fig, ax = plt.subplots(figsize=(14, 5))
avg_returns.plot(ax=ax, color='steelblue', linewidth=0.8)
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Average Daily Returns — Equal Weighted')
ax.set_ylabel('Average Daily Return')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# PLOT 2: Average return per year — equal vs cap weighted
# ============================================================

avg_returns.index    = pd.to_datetime(avg_returns.index)
yearly_equal         = avg_returns.groupby(avg_returns.index.year).mean()

# Cap weighted average returns
common_ret_tickers   = [t for t in returns.columns if t in market_caps.index]
weights_ret          = market_caps[common_ret_tickers]
weights_ret          = weights_ret / weights_ret.sum()
avg_returns_capw     = returns[common_ret_tickers].dot(weights_ret)
avg_returns_capw.index = pd.to_datetime(avg_returns_capw.index)
yearly_capw          = avg_returns_capw.groupby(avg_returns_capw.index.year).mean()

x     = np.arange(len(yearly_equal))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width/2, yearly_equal.values,
       width, color='steelblue', alpha=0.8, label='Equal Weighted')
ax.bar(x + width/2, yearly_capw.values,
       width, color='darkblue', alpha=0.8, label='Cap Weighted')
ax.set_xticks(x)
ax.set_xticklabels(yearly_equal.index, rotation=45)
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Average Daily Return per Year — Equal vs Cap Weighted')
ax.set_ylabel('Average Daily Return')
ax.set_xlabel('Year')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# Computing results used to comment the main statistics

# 1) Mean vs median
mean_minus_median = desc_stats["Mean"] - desc_stats["Median"]

print("Stocks with Mean > Median:", (mean_minus_median > 0).sum())
print("Stocks with Mean < Median:", (mean_minus_median < 0).sum())
print("Average (Mean - Median):", mean_minus_median.mean())


# 2) MAD vs Std Dev
ratio_std_mad = desc_stats["Std Dev"] / desc_stats["MAD"]

print("\nAverage Std/MAD ratio:", ratio_std_mad.mean())
print("Median Std/MAD ratio:", ratio_std_mad.median())


# 3) Skewness and kurtosis
print("\nAverage skewness:", desc_stats["Skewness"].mean())
print("Median skewness:", desc_stats["Skewness"].median())

print("\nAverage kurtosis:", desc_stats["Kurtosis"].mean())
print("Median kurtosis:", desc_stats["Kurtosis"].median())


# PLOT 3: Median vs MAD
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(desc_stats["Median"], desc_stats["MAD"], alpha=0.6)

# Compute correlation
corr = desc_stats["Median"].corr(desc_stats["MAD"])

ax.set_title(f"Median vs MAD (corr = {corr:.2f})")
ax.set_xlabel("Median Daily Return")
ax.set_ylabel("MAD")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot 4: Mean vs Standard Deviation
# ============================================================

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(desc_stats["Mean"], desc_stats["Std Dev"], alpha=0.6)

# Compute correlation
corr = desc_stats["Mean"].corr(desc_stats["Std Dev"])

# Labels and title
ax.set_title(f"Mean vs Standard Deviation (corr = {corr:.2f})")
ax.set_xlabel("Mean Daily Return")
ax.set_ylabel("Standard Deviation")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Correlation between Mean and Std Dev:", corr)

# Equal-weighted vs Cap-weighted average yearly return
print(summary[["Mean", "Median", "MAD", "Std Dev"]])


The descriptive statistics provide a first picture of the basic properties of stock returns and help identify some broad stylized facts present in the sample. In selecting which statistics to report, we also explored a wider set of possible measures suggested by AI, and retained only those that seemed the most noticeable to us.

1. The mean value is on average lower than the median value. We did not expect this, but it is plausible that the reason lies in the presence of some large negative return realizations that are not fully compensated by more regular positive returns. The negative value of skewness is consistent with this interpretation. This asymmetry may also have a psychological component: investors typically fear losses more than they value gains, so drawdowns can trigger stronger selling pressure, whereas there is no equally strong “fear of gains” generating a symmetric effect on the upside.

2. SD is higher than MAD, suggesting that the measure we are about to use in the following steps is more robust for analyzing the behavior of returns. Indeed, MAD grows linearly with the distance from the center, while SD is based on squared deviations, so it grows quadratically. As a consequence, extreme values receive much more weight in SD than in MAD.

3. As shown in plot 3, where MAD and median returns are plotted, there is only a weak correlation between MAD and median return. We did not expect that, since we thought there might be a positive relation similar to the one often discussed between SD and mean returns. A possible reason is that the median is less influenced by outliers, whereas investors should care about extreme losses and gains. A large loss remains economically relevant even if it is statistically extreme. So our interpretation is that we do not observe a strong positive relation because these statistics capture different behavioral patterns.

We also added the simple and capitalization-weighted average returns by year and, quite interestingly:

1. For 2018 and 2022 the average return is very small or negative. This seems consistent with the broader economic and geopolitical context. For 2018, a possible explanation is the beginning of the USA–China trade war, which increased uncertainty and weighed on markets. For 2022, the decline can plausibly be linked not only to the invasion of Ukraine, but also to the (partially consequent) high inflation and the sharp increase in interest rates.
2. Returns for the weighted average are always higher in absolute value than the simple average return. This suggests that large firms experience larger aggregate movements than small firms.



In [ ]:
# ============================================================
# 3.1) more interesting data analysis
# ============================================================


# ============================================================
# Average return of each stock over time and over stocks
# ============================================================
stock_mean_returns = returns.mean(axis=0)    # One value per stock
market_daily_returns = returns.mean(axis=1)  # One value per day


# Helpers to plot distribution + normality tests
def plot_and_test_normality(series, title, xlabel):
    series = pd.Series(series).dropna()

    # Histogram + fitted normal
    fig, ax = plt.subplots(figsize=(14, 5))
    series.hist(
        ax=ax, bins=80, density=True,
        color='steelblue', alpha=0.7,
        label='Empirical distribution'
    )

    mu, sigma = series.mean(), series.std(ddof=1)
    x = np.linspace(series.min(), series.max(), 300)
    ax.plot(
        x, stats.norm.pdf(x, mu, sigma),
        color='tomato', linewidth=2,
        label='Normal distribution'
    )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Descriptive stats
    print(f"N observations: {len(series)}")
    print(f"Skewness: {series.skew():.4f}  (normal = 0)")
    print(f"Kurtosis: {series.kurt():.4f}  (normal = 0)")

    # Tests
    jb_stat, jb_pval = stats.jarque_bera(series)

    ks_stat, ks_pval = stats.kstest(
        series, 'norm', args=(mu, sigma)
    )

    lf_stat, lf_pval = lilliefors(series)

    # Summary table
    rows = [
        ['Jarque-Bera', f'{jb_stat:.4f}', f'{jb_pval:.6f}',
         'Yes' if jb_pval < 0.05 else 'No'],

        ['Kolmogorov-Smirnov*', f'{ks_stat:.4f}', f'{ks_pval:.6f}',
         'Yes' if ks_pval < 0.05 else 'No'],

        ['Lilliefors', f'{lf_stat:.4f}', f'{lf_pval:.6f}',
         'Yes' if lf_pval < 0.05 else 'No'],
    ]

    summary = pd.DataFrame(
        rows,
        columns=['Test', 'Statistic', 'p-value', 'Reject Normality?']
    )
    display(summary)

    print("* KS test uses estimated mean and std → may over-reject normality\n")

# Run for both series
# ============================================================
print("=" * 60)
print("SERIES 1: Average market return per day")
print("=" * 60)
plot_and_test_normality(
    market_daily_returns,
    title  = 'Distribution of Average Market Return per Day',
    xlabel = 'Average Daily Market Return'
)

print("=" * 60)
print("SERIES 2: Average return per stock over time")
print("=" * 60)
plot_and_test_normality(
    stock_mean_returns,
    title  = 'Distribution of Mean Daily Return per Stock',
    xlabel = 'Mean Daily Return per Stock'
)



# ============================================================
# Principal Component Analsysis
# ============================================================

# Compute PCA
pca         = PCA()
pca.fit(returns)

# Variance explained by each component
explained_variance = pca.explained_variance_ratio_

# PLOT 1: Scree plot — how many factors matter?
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Individual variance explained
axes[0].bar(range(1, 21), explained_variance[:20] * 100,
            color='steelblue', alpha=0.8)
axes[0].set_title('Variance Explained by each Component (first 20)')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].grid(True, alpha=0.3)

# Cumulative variance explained
axes[1].plot(range(1, 51),
             np.cumsum(explained_variance[:50]) * 100,
             color='steelblue', linewidth=2, marker='o', markersize=3)
axes[1].axhline(y=50, color='tomato', linestyle='--',
                linewidth=1, label='50% threshold')
axes[1].axhline(y=80, color='green', linestyle='--',
                linewidth=1, label='80% threshold')
axes[1].set_title('Cumulative Variance Explained (first 50 components)')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance Explained (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"PC1 explains: {explained_variance[0]*100:.2f}% of total variance")
print(f"PC2 explains: {explained_variance[1]*100:.2f}% of total variance")
print(f"First 5 PCs explain: {sum(explained_variance[:5])*100:.2f}% of total variance")

# Extract PC1 loads
pc1_loadings = pd.Series(pca.components_[0], index=returns.columns)
scores = pca.transform(returns)
pc1_factor = pd.Series(scores[:, 0], index=returns.index)

# PLOT 2: PC1 over time
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))
pc1_factor.plot(ax=ax, color='steelblue', linewidth=0.8)
ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('PC1 — Market Factor over Time')
ax.set_ylabel('PC1 Score')
ax.set_xlabel('Date')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Rolling average correlation between PC1 score and stock returns
# ============================================================
window = 60

# For each day t, compute correlation between PC1 score and each stock
# over the rolling window, then average across stocks
rolling_corr = pd.DataFrame(index=returns.index, columns=returns.columns, dtype=float)

for ticker in returns.columns:
    rolling_corr[ticker] = returns[ticker].rolling(window).corr(pc1_factor)

# Average across all stocks for each day
avg_rolling_corr = rolling_corr.mean(axis=1)

# PLOT 3: Average rolling correlation
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(avg_rolling_corr.index, avg_rolling_corr.values,
        color='steelblue', linewidth=0.8, label=f'Avg rolling correlation (window={window}d)')

ax.axhline(y=avg_rolling_corr.mean(), color='tomato', linewidth=1.2,
           linestyle='--', label=f'Full-period mean: {avg_rolling_corr.mean():.2f}')

# Shade the area under the curve for visual impact
ax.fill_between(avg_rolling_corr.index, avg_rolling_corr.values,
                avg_rolling_corr.mean(), alpha=0.15, color='steelblue')

# Annotate known crisis periods
crises = {
    'COVID crash\n(Mar 2020)' : '2020-03-16',
    'Low PC1 correlation\n(Nov 2017)' : '2017-11-29',
    }
for label, date in crises.items():
    xval = pd.Timestamp(date)
    if xval in avg_rolling_corr.index:
        yval = avg_rolling_corr[xval]
        ax.annotate(label,
                    xy=(xval, yval),
                    xytext=(30, -30), textcoords='offset points',
                    fontsize=8, color='darkred',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_title(f'Average Rolling Correlation between PC1 and Stock Returns ({window}-day window)')
ax.set_xlabel('Date')
ax.set_ylabel('Average correlation')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics
print(f"Full-period mean correlation : {avg_rolling_corr.mean():.3f}")
print(f"Standard deviation           : {avg_rolling_corr.std():.3f}")
print(f"Max (most systemic day)      : {avg_rolling_corr.max():.3f}  on {avg_rolling_corr.idxmax().date()}")
print(f"Min (most idiosyncratic day) : {avg_rolling_corr.min():.3f}  on {avg_rolling_corr.idxmin().date()}")


# ============================================================
# Rolling 60-day correlation between Tech and Financials — 2017
# ============================================================

start = '2017-07-01'
end   = '2018-03-31'

# Get sector tickers
sp500_info = sp500[['Symbol', 'GICS Sector']].copy()
sp500_info['Symbol'] = sp500_info['Symbol'].str.replace('.', '-', regex=False)

tech_tickers = sp500_info[sp500_info['GICS Sector'] == 'Information Technology']['Symbol'].tolist()
fin_tickers  = sp500_info[sp500_info['GICS Sector'] == 'Financials']['Symbol'].tolist()

tech_tickers = [t for t in tech_tickers if t in returns.columns]
fin_tickers  = [t for t in fin_tickers  if t in returns.columns]

# Equal-weighted sector daily returns
tech_daily = returns.loc[start:end, tech_tickers].mean(axis=1)
fin_daily  = returns.loc[start:end, fin_tickers].mean(axis=1)

# Rolling 60-day correlation between the two sector return series
rolling_corr = tech_daily.rolling(60).corr(fin_daily)

# Plot
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(rolling_corr.index, rolling_corr.values,
        color='steelblue', linewidth=1.2,
        label='60-day rolling correlation: Tech vs Financials')

ax.axhline(rolling_corr.mean(), color='gray', linewidth=1,
           linestyle='--', label=f'Mean: {rolling_corr.mean():.2f}')

ax.axvline(pd.Timestamp('2017-11-29'), color='tomato', linewidth=1.2,
           linestyle='--', label='Nov 29 rotation')

ax.set_title('Rolling 60-day Correlation — Technology vs Financials (2017)')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling correlation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Correlation on Nov 29: {rolling_corr.loc['2017-11-29']:.3f}")
print(f"Full period mean     : {rolling_corr.mean():.3f}")
print(f"Full period min      : {rolling_corr.min():.3f} on {rolling_corr.idxmin().date()}")


# PLOT 4: Scatter — PC1 loading vs variance per stock
# ============================================================

# Find the stock with the highest variance and add it to notable
highest_var_ticker = variance.idxmax()
print(f"Stock with highest variance: {highest_var_ticker} ({variance[highest_var_ticker]:.6f})")

notable = ['AAPL', 'MSFT', 'AIG', 'NVDA', 'AMZN', 'GOOGL', 'META', 'TSLA', highest_var_ticker]
notable = list(dict.fromkeys(notable))

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(pc1_loadings, variance,
           color='steelblue', alpha=0.6, s=30)

for ticker in notable:
    if ticker in pc1_loadings.index:
        ax.annotate(ticker,
                    xy=(pc1_loadings[ticker],
                        variance[ticker]),
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8, color='darkred')

ax.set_title('PC1 Loading vs Variance per Stock')
ax.set_xlabel('Loading on PC1')
ax.set_ylabel('Variance of stock returns')
ax.axhline(y=0, color='black', linewidth=0.5, linestyle='--')
ax.axvline(x=0, color='black', linewidth=0.5, linestyle='--')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Top and bottom 5 stocks by PC1 loading
print("\nTop 5 stocks with highest PC1 loadings:")
print(pc1_loadings.nlargest(5))

print("\nTop 5 stocks with lowest PC1 loadings:")
print(pc1_loadings.nsmallest(5))



# SMCI daily returns over time
# ============================================================
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(returns.index, returns['SMCI'],
        color='steelblue', linewidth=0.8, label='SMCI daily return')

ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')

# Mark the top 5 spikes (positive and negative) so they are easy to spot
top_pos = returns['SMCI'].nlargest(5)
top_neg = returns['SMCI'].nsmallest(5)

ax.scatter(top_pos.index, top_pos.values, color='green', zorder=5, s=40, label='Top 5 positive spikes')
ax.scatter(top_neg.index, top_neg.values, color='red',   zorder=5, s=40, label='Top 5 negative spikes')

# Annotate each spike with its date and value
for date, val in pd.concat([top_pos, top_neg]).items():
    ax.annotate(f"{date.strftime('%Y-%m-%d')}\n{val:.1%}",
                xy=(date, val),
                xytext=(10, 10 if val > 0 else -25),
                textcoords='offset points',
                fontsize=7, color='darkred',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

ax.set_title('SMCI — Daily Returns over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Return')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()




# ============================================================
# Hierarchical clustering on the return correlation matrix
# ============================================================

# compute correlation matrix and convert to distances
corr_matrix = returns.corr()
distance_matrix = 1 - corr_matrix # Totally suggested by AI but it makes sense. if corr = 1 distance is 0 and the maximum is reached if the two stocks move in the completely different direction

condensed_dist = squareform(distance_matrix, checks=False)  # we condense to only take unique values,also scipy needs condensed matrix to work

# hierarchical clustering (Ward minimizes within-cluster variance)
linkage_matrix = linkage(condensed_dist, method='ward')
# linkage_matrix = linkage(condensed_dist, method='average') we tried it at first following the material in A03b but it gave us very unblacend cluster (also for big number of clusters) so we used 'ward' followin AI suggestion even if ward its assuming euclidean distances which is not our case


# Plot 1: Dendrogram (truncated — showing only the top levels is cleaner)
# ============================================================
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(
    linkage_matrix,
    labels=returns.columns.tolist(),
    leaf_rotation=90,
    leaf_font_size=4,
    truncate_mode='lastp',
    p=461,
    ax=ax
)
ax.set_title('Hierarchical Clustering Dendrogram — S&P 500 stocks')
ax.set_xlabel('Stock / cluster')
ax.set_ylabel('Distance (Ward linkage)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


# Davies-Bouldin score
# ============================================================
db_scores = []
k_range = range(5, 15)

for k in k_range:
    labels = fcluster(linkage_matrix, k, criterion='maxclust')
    score = davies_bouldin_score(returns.T, labels)
    db_scores.append(score)

plt.figure(figsize=(10, 4))
plt.plot(k_range, db_scores, marker='o', color='steelblue')
plt.xlabel('Number of clusters k')
plt.ylabel('Davies-Bouldin score')
plt.title('Davies-Bouldin score by number of clusters')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_range[np.argmin(db_scores)]
print(f"Best k by Davies-Bouldin: {best_k}")


# Elbow method — within-cluster distance vs k
# ============================================================
k_range = range(2, 20)
within_var = []

for k in k_range:
    labels = fcluster(linkage_matrix, k, criterion='maxclust')
    wv = 0
    for c in np.unique(labels):
        members = returns.columns[labels == c]
        # sum all pairwise distances within the cluster
        sub = distance_matrix.loc[members, members].values
        wv += sub.sum() / 2
    within_var.append(wv)

# Compute the "elbow" analytically via the second derivative
deltas = np.diff(within_var)           # we simply compute how mcuh total within-cluster distance decrease by allowing for one more cluster
delta2 = np.diff(deltas)               # we compute the second difference
elbow_k = list(k_range)[np.argmin(delta2) + 2]  # +2 to account for diff offset
print(f"Elbow detected at k = {elbow_k}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw within-cluster distance
axes[0].plot(k_range, within_var, marker='o', color='steelblue', linewidth=1.5)
axes[0].axvline(x=elbow_k, color='tomato', linestyle='--',
                linewidth=1.2, label=f'Elbow at k={elbow_k}')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Total within-cluster distance')
axes[0].set_title('Elbow method')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Right: rate of decrease (first differences)
axes[1].plot(list(k_range)[1:], np.abs(deltas), marker='o',
             color='steelblue', linewidth=1.5)
axes[1].axvline(x=elbow_k, color='tomato', linestyle='--',
                linewidth=1.2, label=f'Elbow at k={elbow_k}')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Decrease in within-cluster distance')
axes[1].set_title('Marginal gain from adding one more cluster')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Cut the tree into k clusters and assign labels
k = 8
cluster_labels = fcluster(linkage_matrix, k, criterion='maxclust')
cluster_series = pd.Series(cluster_labels, index=returns.columns, name='cluster')

print(f"\nCluster sizes (k={k}):")
print(cluster_series.value_counts().sort_index())


# compare clusters to actual sectors
sp500_info = sp500[['Symbol', 'GICS Sector']].copy()
sp500_info['Symbol'] = sp500_info['Symbol'].str.replace('.', '-', regex=False)
sp500_info = sp500_info.set_index('Symbol')

cluster_df = cluster_series.to_frame().join(sp500_info, how='left')

print(f"\nSector composition per cluster (k={k}):")
sector_crosstab = pd.crosstab(cluster_df['cluster'], cluster_df['GICS Sector'])
display(sector_crosstab)


# Rank GICS sectors by how concentrated they are across clusters
# ============================================================
# For each sector, we compute how spread its stocks are across clusters
sector_concentration = []

for sector in cluster_df['GICS Sector'].dropna().unique():
    sector_stocks = cluster_df[cluster_df['GICS Sector'] == sector]
    n_stocks = len(sector_stocks)

    # how many distinct clusters does this sector appear in?
    n_clusters_used = sector_stocks['cluster'].nunique()

    # what fraction of stocks are in the dominant cluster?
    dominant_cluster_frac = sector_stocks['cluster'].value_counts().iloc[0] / n_stocks

    # entropy
    cluster_counts = sector_stocks['cluster'].value_counts(normalize=True)
    entropy = -np.sum(cluster_counts * np.log(cluster_counts))

    sector_concentration.append({
        'Sector'                : sector,
        'N stocks'              : n_stocks,
        'N clusters used'       : n_clusters_used,
        'Dominant cluster %'    : round(dominant_cluster_frac * 100, 1),
        'Entropy (lower=better)': round(entropy, 3),
    })

concentration_df = pd.DataFrame(sector_concentration)

# Sort by entropy ascending
concentration_df = concentration_df.sort_values('Entropy (lower=better)').reset_index(drop=True)

print("Sectors ranked by concentration across clusters (least spread first):")
display(concentration_df)

To finish the third section of the assignment on descriptive statistics, we tried some more in-depth analysis.

## Distribution of returns
At first, we tried to plot the distribution of daily average returns in the market and then the average return for each stock observed across the full period. Two interesting facts emerge:

1. The average daily returns seem to be distributed like a normal distribution, but actually they are not, probably because of fat tails and negative skewness, as suggested by the three tests we carried out. We did not know the Lilliefors test, and it was suggested by AI when we asked whether the other two tests were appropriate. In particular, we learned that Lilliefors is more appropriate than K-S in this case, since, when comparing the actual CDF with the hypothetical CDF, we use the mean and standard deviation estimated from the data itself.

2. At the beginning, we asked the AI why the distribution of daily returns was normal. This was the wrong question, since it was not normal. The AI told us that it was because of the CLT. Interestingly, the AI forgot that this argument is not valid here because average daily returns are unlikely to be i.i.d.

## PCA
Secondly, we used PCA to capture the underlying factors in market returns and to see which stocks had the highest and lowest loadings.

1. Big tech firms have fairly large loadings, but they are not the main contributors. Most of the companies with the highest loadings are in the financial or insurance sector, such as BlackRock, since those companies heavily depend on interest rates and the business cycle. On the other hand, Kroger and Campbell Soup (supermarkets/food) and Newmont (gold mining) are among the companies with the lowest loadings, reflecting the idea that gold is sometimes countercyclical and that, regardless of how the economy is performing, people buy more or less the same amount of food.

2. We plotted the rolling correlation between PC1 and stock returns, and it is interesting to observe where the lowest and highest peaks occurred. The highest peak in correlation was in March 2020, when the pandemic started, suggesting that the fall in returns across stocks in that month was largely due to a common underlying factor. On the contrary, the lowest correlation was in November 2017. We asked AI the reason for this finding, and it gave us an explanation based on changes in interest rates that shifted resources from the technology sector to the financial sector. However, to partially check the validity of this explanation, we looked at the rolling correlation of returns between the two sectors. The graph suggests that what AI told us could have been true for the months after November 2017, but not for the exact moment we are interested in, since up to that point the correlation between the two sectors remained quite high.

3. Interestingly, in the PCA of A03c, AIG emerged because of its high variance but modest contribution to the PCA. Here, this anomaly is no longer so evident, since we are considering a different period of time. On the contrary, SMCI resulted as an outlier due to some accounting scandals: in 2018 it failed to file its financial statements for two years in a row, causing its delisting from Nasdaq, and later benefited from the AI boom, since the company provides servers and storage systems.

## Cluster analysis
We also attempted a cluster analysis to see whether there was any specific relation between sectors in terms of return correlations.

Before discussing the results, we report that AI was very useful in this case, providing us with better measures of distance and agglomeration methods than those we had initially implemented. It also provided us with several methods to choose the optimal number of clusters.

We tried two of them and, in the end, chose to use 8 clusters. Indeed, the Davies-Bouldin method suggested using too many clusters. Since we wanted to see whether some of the 11 sectors could be grouped together, using more clusters than sectors was not very useful. On the contrary, the elbow method suggested, as shown in the graph, that after 8 clusters there are no significant improvements in the decrease of within-group distance.

Then, we ranked sectors based on how dispersed they were across the 8 clusters. Interestingly, materials and energy are among the least dispersed sectors, since they depend on their own specific factors. This is especially understandable in the case of energy. On the contrary, the communication sector is very dispersed. A possible explanation is that this sector contains very different companies, from Meta and Netflix to more traditional communication companies.



In [ ]:
# ============================================================
# GROUPING WitH AUTOENCODERS
# ============================================================

# Annual returns — compounded daily returns
annual_returns = (
    (1 + returns)
    .resample('YE')
    .prod()
    - 1
)

# Annual volatility — std of daily returns within each year
annual_vol = (
    returns
    .resample('YE')
    .std()
)

# Annual skewness
annual_skew = (
    returns
    .resample('YE')
    .apply(lambda x: x.skew())
)

# Annual Sharpe ratio
annual_sharpe = annual_returns / (annual_vol + 1e-8)

print(f"Annual returns shape:  {annual_returns.shape}")
print(f"Annual vol shape:      {annual_vol.shape}")
print(f"Annual skew shape:     {annual_skew.shape}")
print(f"Annual sharpe shape:   {annual_sharpe.shape}")


# Stack all features per stock
def to_stock_matrix(df, suffix):
    df = df.dropna()
    df.columns = [f'{col}' for col in df.columns]
    transposed = df.T
    transposed.columns = [f'{suffix}_y{i+1}' for i in range(transposed.shape[1])]
    return transposed

ret_df  = to_stock_matrix(annual_returns, 'ret')
vol_df  = to_stock_matrix(annual_vol,     'vol')
skew_df = to_stock_matrix(annual_skew,    'skew')
sh_df   = to_stock_matrix(annual_sharpe,  'sharpe')

tickers_common = ret_df.index.intersection(vol_df.index) \
                             .intersection(skew_df.index) \
                             .intersection(sh_df.index)

feature_df = pd.concat(
    [ret_df.loc[tickers_common],
     vol_df.loc[tickers_common],
     skew_df.loc[tickers_common],
     sh_df.loc[tickers_common]],
    axis=1
).dropna()

tickers = feature_df.index
data    = feature_df.values.astype(np.float32)

n_stocks, n_features = data.shape
print(f"\nFinal dataset shape: {data.shape}")
print(f"  → {n_stocks} stocks × {n_features} features")
print(f"  → {len(ret_df.columns)} years × 4 features per year")


# Split train and validatin, then standardise
Xtrain, Xval, tickers_train, tickers_val = train_test_split(
    data, tickers,
    test_size    = 0.2,
    random_state = 42,
    shuffle      = True
)

scaler      = StandardScaler()
Xtrain      = scaler.fit_transform(Xtrain)
Xval        = scaler.transform(Xval)
data_scaled = scaler.transform(data)

print(f"\nTrain samples: {len(Xtrain)}  |  Val samples: {len(Xval)}")


# Build the autoencoder
model = Sequential(name='autoencoder')
model.add(Input(shape=(n_features,)))

model.add(Dense(20, activation='tanh', kernel_initializer=GlorotUniform(), name='enc_2'))
model.add(Dense(3,  activation='linear',                                     name='bottleneck'))
model.add(Dense(20,         activation='tanh',   kernel_initializer=GlorotUniform(), name='dec_1'))
model.add(Dense(n_features, activation='linear',                                     name='output'))

model.summary()


# Compile and train
def r2(y_true, y_pred):
    return (
        1 - tf.reduce_mean(tfp.stats.variance(y_pred - y_true))
          / tf.reduce_mean(tfp.stats.variance(y_true))
    )

model.compile(optimizer='adam', loss='mse', metrics=[r2])


early_stop = EarlyStopping(
    monitor              = 'val_loss',
    mode                 = 'min',
    patience             = 10,
    restore_best_weights = True
)

reduce_lr = ReduceLROnPlateau(
    monitor  = 'val_loss',
    mode     = 'min',
    factor   = 0.5,
    patience = 5,
    verbose  = 1
)

history = model.fit(
    Xtrain, Xtrain,
    validation_data = (Xval, Xval),
    epochs          = 200,
    batch_size      = 32,
    callbacks       = [early_stop, reduce_lr],
    verbose         = 1
)

best_epoch = np.argmin(history.history['val_loss']) + 1
print(f"Best epoch:    {best_epoch}")
print(f"Best val loss: {min(history.history['val_loss']):.6f}")
print(f"Best val R²:   {history.history['val_r2'][best_epoch-1]:.4f}")


# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'],     label='Train', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val',   color='tomato')
axes[0].set_title('Reconstruction Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['r2'],     label='Train', color='steelblue')
axes[1].plot(history.history['val_r2'], label='Val',   color='tomato')
axes[1].axhline(y=1, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('R² Score')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

encoder = tf.keras.Model(
    inputs=model.inputs,
    outputs=model.get_layer('bottleneck').output
)

Z = encoder.predict(data_scaled)

latent_df = pd.DataFrame(
    Z,
    index=tickers,
    columns=['z1', 'z2', 'z3']
)

print(latent_df.head(3))

We attempted to repeat the cluster analysis using autoencoders, but we encountered several issues. In our initial attempts, we used a dataset consisting of daily returns for each stock over a 10-year period. However, the models trained on this dataset delivered very poor results, with R^2
 values below 0.2.

We experimented with several specifications—changing the architecture, activation functions, and adding regularization techniques such as dropout and L2—but observed only marginal improvements. We initially suspected that this might be due to the fact that the number of features (over 2,700) greatly exceeds the number of observations (fewer than 500).

However, in the dataset used in class for movie classification, a similar imbalance between variables and observations is present, yet the autoencoders performed much better. This led us to hypothesize that the poor performance in our case may instead be due to the extremely noisy nature of daily returns, which makes it difficult for the autoencoder to extract meaningful patterns.

To address this issue, we first tried using weekly returns, and then semi-annual returns, in order to reduce noise and decrease the dimensionality of the dataset. This led to visible, but still limited, improvements.

Finally, we adopted the approach presented here. Specifically, we used yearly returns and augmented them with additional statistics—such as volatility—that are derived from returns and may help the autoencoder capture more meaningful structure in the data. With this final specification, we were able to train models achieving R^2 values above 0.3, which, although still relatively modest, represented a substantial improvement over the initial specifications.

Although the results remain below our expectations,especially when compared to our benchmark (the movie model), we believe this may be due to the inherently weaker and less stable patterns underlying the data-generating process of financial returns, which are much less consistent than those observed in movie preferences.

In all these attempts, AI was very helpful. We described to the AI the behavior of the loss function on the training and validation sets, as well as the distribution of gradients. We then brainstormed with the AI to understand the possible causes of the anomalies we observed and to identify which hyperparameter choices could help improve the model.

In [ ]:
# Join latent space with GICS sectors and original dataset
sp500_info = sp500[['Symbol', 'GICS Sector']].copy()
sp500_info['Symbol'] = sp500_info['Symbol'].str.replace('.', '-', regex=False)
sp500_info = sp500_info.set_index('Symbol')

z_cols  = [c for c in latent_df.columns if c.startswith('z')]
plot_df = latent_df[z_cols].join(sp500_info, how='left').reset_index()
plot_df.columns = ['Ticker'] + z_cols + ['Sector']
plot_df['Sector'] = plot_df['Sector'].fillna('Unknown')

# Helper function
def make_3d_plot(df, color_col, title):
    fig = px.scatter_3d(
        df,
        x          = 'z1',
        y          = 'z2',
        z          = 'z3',
        color      = color_col,
        hover_name = 'Ticker',
        hover_data = {'z1': ':.3f', 'z2': ':.3f', 'z3': ':.3f', 'Sector': True},
        title      = title,
        opacity    = 0.8,
        height     = 750,
    )
    fig.update_traces(marker=dict(size=4))
    fig.update_layout(
        legend = dict(title=color_col, font=dict(size=10)),
        scene  = dict(xaxis_title='z1', yaxis_title='z2', zaxis_title='z3')
    )
    return fig
Z_values = latent_df.values

# ════════════════════════════════════════════════════════════
# 1. Plot all sectors in the 3D space generated by the bottleneck
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("PLOT 1: All sectors")
print("=" * 60)

make_3d_plot(plot_df, 'Sector', 'Latent Space — All Sectors').show()

# there are too many sectors so that it is not very clear whether the 3 components are actually allowing us to seprate across sectors or not. Let's focus on 3 sectors that should be sufficiently different from each other

# ════════════════════════════════════════════════════════════
# 2. Focus on Energy, Financials, Consumer Staples only
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("PLOT 2: Energy, Financials, Consumer Staples")
print("=" * 60)

sectors_of_interest = ['Energy', 'Financials', 'Consumer Staples']
plot_df_filtered    = plot_df[plot_df['Sector'].isin(sectors_of_interest)].copy()

make_3d_plot(
    plot_df_filtered, 'Sector',
    'Latent Space — Energy, Financials & Consumer Staples'
).show()

print(f"Stocks in filtered plot: {len(plot_df_filtered)}")
print(plot_df_filtered['Sector'].value_counts())


# ════════════════════════════════════════════════════════════
# 3. Clustering + elbow curve
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("CLUSTERING + ELBOW")
print("=" * 60)


condensed = pdist(Z_values, metric='euclidean')
Z_link    = linkage(condensed, method='ward')

k_range    = range(2, 20)
within_var = []
db_scores  = []

for k in k_range:
    labels = fcluster(Z_link, k, criterion='maxclust')

    # Within-cluster distance
    wv = 0
    for c in np.unique(labels):
        members  = np.where(labels == c)[0]
        sub      = Z_values[members]
        centroid = sub.mean(axis=0)
        wv      += ((sub - centroid) ** 2).sum()
    within_var.append(wv)

    # Davies-Bouldin
    db_scores.append(davies_bouldin_score(Z_values, labels))

# Elbow detection
deltas  = np.diff(within_var)
delta2  = np.diff(deltas)
elbow_k = list(k_range)[np.argmin(delta2) + 2]
best_k_db = list(k_range)[np.argmin(db_scores)]

print(f"Elbow detected at k      = {elbow_k}")
print(f"Best k by Davies-Bouldin = {best_k_db}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, within_var, marker='o', color='steelblue', linewidth=1.5)
axes[0].axvline(x=elbow_k, color='tomato', linestyle='--',
                linewidth=1.2, label=f'Elbow at k={elbow_k}')
axes[0].set_title('Elbow Method — Within-cluster Distance')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Total within-cluster distance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, db_scores, marker='o', color='steelblue', linewidth=1.5)
axes[1].axvline(x=best_k_db, color='tomato', linestyle='--',
                linewidth=1.2, label=f'Best k={best_k_db}')
axes[1].set_title('Davies-Bouldin Score')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Davies-Bouldin score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# ════════════════════════════════════════════════════════════
# 4. Cut at k=8 sector composition + concentration analysis
# ════════════════════════════════════════════════════════════
print("=" * 60)
print("SECTOR COMPOSITION — k=8 clusters")
print("=" * 60)

k = 8
cluster_labels = fcluster(Z_link, k, criterion='maxclust')

# Attach cluster labels to plot_df for 3D visualisation
plot_df['AE Cluster'] = cluster_labels.astype(str)

make_3d_plot(
    plot_df, 'AE Cluster',
    f'Latent Space — {k} Clusters (Ward linkage)'
).show()

# Attach cluster labels to ticker index for sector analysis
cluster_series = pd.Series(cluster_labels, index=latent_df.index, name='cluster')
cluster_df     = cluster_series.to_frame().join(sp500_info, how='left')

print(f"\nCluster sizes (k={k}):")
print(cluster_series.value_counts().sort_index())

print(f"\nSector composition per cluster (k={k}):")
sector_crosstab = pd.crosstab(cluster_df['cluster'], cluster_df['GICS Sector'])
display(sector_crosstab)

# Sector concentration analysis
sector_concentration = []

for sector in cluster_df['GICS Sector'].dropna().unique():
    sector_stocks = cluster_df[cluster_df['GICS Sector'] == sector]
    n_stocks      = len(sector_stocks)

    n_clusters_used       = sector_stocks['cluster'].nunique()
    dominant_cluster_frac = sector_stocks['cluster'].value_counts().iloc[0] / n_stocks

    cluster_counts = sector_stocks['cluster'].value_counts(normalize=True)
    entropy        = -np.sum(cluster_counts * np.log(cluster_counts))

    sector_concentration.append({
        'Sector'                 : sector,
        'N stocks'               : n_stocks,
        'N clusters used'        : n_clusters_used,
        'Dominant cluster %'     : round(dominant_cluster_frac * 100, 1),
        'Entropy' : round(entropy, 3),
    })

concentration_df = pd.DataFrame(sector_concentration)
concentration_df = concentration_df.sort_values('Entropy').reset_index(drop=True)

print("\nSectors ranked by concentration across clusters (least spread first):")
display(concentration_df)


Since we applied our autoencoder to a dataset with a relatively small number of features per observation (due to our decision to significantly reduce dimensionality), we were able to train a model with only three neurons in the bottleneck layer. This allowed us to easily visualize the observations in a three-dimensional space based on their values along these three latent components.

Plotting the stocks and coloring them by sector suggests that the autoencoder was indeed able to capture some underlying factors that are partially related to economic sectors.

We also attempted to determine the optimal number of clusters using the same two methods as before (elbow method and Davies-Bouldin index). Although these methods did not indicate that k=8 was optimal, we nevertheless chose to retain 8 clusters in order to facilitate comparison with the previous clustering results.

Interestingly, this new approach produces clusters that align less closely with the existing sector classification. Moreover, when ranking sectors by entropy, we observe a different pattern compared to the previous method. In particular, while Energy and Utilities still exhibit relatively low entropy (indicating concentration within clusters), Materials displays significantly higher entropy.

At this stage, we are not able to definitively determine which of the two clustering approaches should be preferred. Although the performance metrics of the autoencoder are not particularly strong, it is important to note that the distance measures used in the two clustering methods are fundamentally different. In the first approach, clustering is entirely based on correlations of returns between stocks. In the second approach, clustering is based on latent representations that reflect broader underlying characteristics, such as aggregated return dynamics and related features that should have been captured by the autoencoder.

For this reason, both methods may be considered valid, despite producing quite different clustering structures. However, we remain somewhat skeptical of the second method, since in several attempts we observed that simply changing the architecture of the neural network resulted in very different cluster compositions.

In [ ]:
# ============================================================
# 4) Divide the sample in train and test subsamples
# ============================================================

np.random.seed(42)
selected_stocks = np.random.choice(returns.columns, size=100, replace=False)
returns_subset  = returns[selected_stocks]

# We do not split at random but based on position since the data are time-based
split_index   = int(len(returns_subset) * 0.8)
split_date    = returns_subset.index[split_index]
returns_train = returns_subset.iloc[:split_index]
returns_test  = returns_subset.iloc[split_index:]

print(f"Split date:   {split_date.strftime('%Y-%m-%d')}")
print(f"Train sample: {returns_train.shape[0]} days")
print(f"Test sample:  {returns_test.shape[0]} days")


In the following sections we tried to construct the optimal portfolios satisfying the conditions indicated in the assignment. In particular, we had to construct 10 portfolios with different target medians and, for each of them, we wanted to minimize the MAD.

This exercise is not as simple as a standard mean-variance optimization, since the median is not a smooth function, making the problem non-convex.

For this reason we used two different procedures.

1. The first procedure is a proxy method. Instead of minimizing the true MAD around the realized median of the portfolio, we minimize the average absolute deviation of portfolio returns around the target median:
$$
\min_w \frac{1}{T}\sum_t |(Rw)_t - m_{\text{target}}|
$$
with the usual constraints
$$
\sum_i w_i = 1, \qquad w_i \geq 0.
$$
This is much easier to solve, because the target median is fixed and the exercise becomes a linear optimization problem. However, this is only an approximation, because the true realized median of the portfolio is not forced to be equal to the target median.

2. The second method tries to solve the real problem. In this case we minimize a loss function containing both the true MAD of the portfolio and a penalty for being far from the desired median:
$$
L(w)=\text{MAD}(w)+\lambda |\text{median}(Rw)-m_{\text{target}}|
$$
where:
$$
\text{MAD}(w)=\frac{1}{T}\sum_t |(Rw)_t-\text{median}(Rw)|.
$$
The parameter λ controls how strongly we force the portfolio median to be close to the target median. To impose positive weights summing to one, we do not optimize the weights directly, but we optimize some unconstrained variables and then transform them with a softmax function. Since this optimization is not convex and can depend a lot on the starting point, we use a multi-start procedure: we start from the proxy portfolio and then we add several random perturbations.
Moreover, the code tries several values of λ. If one solution reaches the target median within the tolerance, we choose the one with the lowest MAD. If no solution reaches the tolerance, we choose the one with the smallest median error. Since this part is quite slow, the code also saves the results and loads them from cache when they are already available.

In [ ]:
# ============================================================
# Step 5: Proxy portfolio and multistart penalized portfolio
# with Drive cache and GitHub fallback
# ============================================================

import os
from pathlib import Path
import urllib.request
import urllib.error

import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.optimize import minimize

# ------------------------------------------------------------
# Optional: mount Google Drive if running in Colab
# ------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
R_full = returns_train.values
T_full, N = R_full.shape

stock_medians  = returns_train.median()
min_median     = stock_medians.min()
max_median     = stock_medians.max()
target_medians = np.linspace(min_median, max_median, 10)

print(f"Min stock median: {min_median:.6f}")
print(f"Max stock median: {max_median:.6f}")

# ------------------------------------------------------------
# Results directory and cache file names
# ------------------------------------------------------------
if IN_COLAB:
    RESULTS_DIR = Path("/content/drive/MyDrive/21046-mlf-ps-results")
else:
    RESULTS_DIR = Path("../results")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/stfgrz/21046-mlf-ps/main/data"

WEIGHTS_FILE      = RESULTS_DIR / "warmstart_multistart_softmax_portfolio_weights.csv"
FRONTIER_FILE     = RESULTS_DIR / "warmstart_multistart_softmax_frontier_summary.csv"
DIAGNOSTICS_FILE  = RESULTS_DIR / "warmstart_multistart_diagnostics.csv"
LAMBDA_FILE       = RESULTS_DIR / "warmstart_multistart_lambda_insample_diagnostics.csv"
COMPARISON_FILE   = RESULTS_DIR / "warmstart_multistart_homogeneous_comparison.csv"
PROXY_FILE        = RESULTS_DIR / "warmstart_multistart_proxy_summary.csv"

FILE_MAP = {
    "warmstart_multistart_softmax_portfolio_weights.csv": WEIGHTS_FILE,
    "warmstart_multistart_softmax_frontier_summary.csv": FRONTIER_FILE,
    "warmstart_multistart_diagnostics.csv": DIAGNOSTICS_FILE,
    "warmstart_multistart_lambda_insample_diagnostics.csv": LAMBDA_FILE,
    "warmstart_multistart_homogeneous_comparison.csv": COMPARISON_FILE,
    "warmstart_multistart_proxy_summary.csv": PROXY_FILE,
}

def ensure_file_from_github(filename, local_path):
    if local_path.exists():
        return True

    url = f"{GITHUB_RAW_BASE}/{filename}"
    try:
        print(f"Trying GitHub: {url}")
        urllib.request.urlretrieve(url, local_path)
        print(f"Downloaded {filename} to {local_path}")
        return True
    except urllib.error.HTTPError:
        return False
    except Exception as e:
        print(f"Error downloading {filename}: {e}")
        return False

# ------------------------------------------------------------
# Tuning parameters
# ------------------------------------------------------------
lambda_grid = [10, 30, 100, 300, 1000, 3000, 10000]
median_tolerance = 1e-5

# ------------------------------------------------------------
# Proxy optimization
# ------------------------------------------------------------
def min_abs_dev_around_target(R, target_median):
    T, N = R.shape

    w = cp.Variable(N)
    u = cp.Variable(T)

    port_returns = R @ w

    objective = cp.Minimize(cp.sum(u) / T)
    constraints = [
        u >= port_returns - target_median,
        u >= -(port_returns - target_median),
        u >= 0,
        cp.sum(w) == 1,
        w >= 0
    ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.HIGHS)

    return w.value, prob.value, prob.status

# ------------------------------------------------------------
# Portfolio utilities
# ------------------------------------------------------------
def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

def weights_to_softmax_logits(w, eps=1e-12):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, eps, None)
    w = w / w.sum()
    return np.log(w)

def compute_true_mad(port_returns):
    med = np.median(port_returns)
    mad = np.mean(np.abs(port_returns - med))
    return med, mad

def compute_target_mad(port_returns, target_median):
    return np.mean(np.abs(port_returns - target_median))

def compute_portfolio_stats_from_weights(R, w, target_median=None):
    port_returns = R @ w
    median, mad = compute_true_mad(port_returns)

    out = {
        "port_returns": port_returns,
        "median": median,
        "mad": mad
    }

    if target_median is not None:
        out["target_mad"] = compute_target_mad(port_returns, target_median)

    return out

def portfolio_stats_from_z(z, R, target_median=None):
    w = softmax(z)
    stats = compute_portfolio_stats_from_weights(R, w, target_median=target_median)
    return w, stats

def penalized_objective(z, R, target_median, penalty):
    _, stats = portfolio_stats_from_z(z, R, target_median=target_median)
    median_error = abs(stats["median"] - target_median)
    return stats["mad"] + penalty * median_error

# ------------------------------------------------------------
# Multistart local optimization
# ------------------------------------------------------------
def generate_multiple_starts(z_proxy, n_starts=8, perturb_scale=2.0, seed=42):
    rng = np.random.default_rng(seed)
    starts = [z_proxy.copy()]

    for _ in range(n_starts - 1):
        eps = rng.normal(loc=0.0, scale=perturb_scale, size=z_proxy.shape)
        z_new = z_proxy + eps
        z_new = z_new - np.mean(z_new)
        starts.append(z_new)

    return starts

def run_one_local_start(z0, R, target_median, penalty, maxiter=20000):
    result = minimize(
        fun=penalized_objective,
        x0=z0,
        args=(R, target_median, penalty),
        method="Powell",
        options={
            "maxiter": maxiter,
            "xtol": 1e-7,
            "ftol": 1e-7
        }
    )

    z_star = result.x
    w_star, stats = portfolio_stats_from_z(z_star, R, target_median=target_median)

    return {
        "weights": w_star,
        "z_star": z_star,
        "median": stats["median"],
        "mad": stats["mad"],
        "target_mad": stats["target_mad"],
        "median_error": stats["median"] - target_median,
        "abs_median_error": abs(stats["median"] - target_median),
        "objective_value": result.fun,
        "success": result.success,
        "message": result.message
    }

def select_best_result(candidates):
    return sorted(
        candidates,
        key=lambda d: (
            d["abs_median_error"],
            d["target_mad"],
            d["mad"],
            d["objective_value"]
        )
    )[0]

def solve_true_mad_portfolio_multistart(
    R,
    target_median,
    z_proxy,
    penalty,
    maxiter=20000,
    n_starts=8,
    perturb_scale=2.0,
    seed=42
):
    starts = generate_multiple_starts(
        z_proxy=z_proxy,
        n_starts=n_starts,
        perturb_scale=perturb_scale,
        seed=seed
    )

    all_results = []
    for k, z0 in enumerate(starts):
        res = run_one_local_start(
            z0=z0,
            R=R,
            target_median=target_median,
            penalty=penalty,
            maxiter=maxiter
        )
        res["start_id"] = k
        res["is_proxy_start"] = (k == 0)
        all_results.append(res)

    best_result = select_best_result(all_results)
    best_result["all_start_results"] = all_results
    return best_result

# ------------------------------------------------------------
# Lambda selection
# ------------------------------------------------------------
def choose_lambda_for_target(
    R,
    target_median,
    lambda_grid,
    median_tolerance=1e-5,
    n_starts=20,
    perturb_scale=2.5,
    maxiter=20000,
    seed=42
):
    proxy_w, proxy_obj, proxy_status = min_abs_dev_around_target(R, target_median)

    if proxy_w is None or proxy_status not in ["optimal", "optimal_inaccurate"]:
        raise RuntimeError(f"Proxy optimization failed for target {target_median:.6f}")

    proxy_w = np.maximum(proxy_w, 0.0)
    proxy_w = proxy_w / proxy_w.sum()
    z_proxy = weights_to_softmax_logits(proxy_w)

    lambda_results = []
    for j, lam in enumerate(lambda_grid):
        fit_res = solve_true_mad_portfolio_multistart(
            R=R,
            target_median=target_median,
            z_proxy=z_proxy,
            penalty=lam,
            maxiter=maxiter,
            n_starts=n_starts,
            perturb_scale=perturb_scale,
            seed=seed + j
        )

        lambda_results.append({
            "lambda": lam,
            "median": fit_res["median"],
            "mad": fit_res["mad"],
            "target_mad": fit_res["target_mad"],
            "median_error": fit_res["median_error"],
            "abs_median_error": fit_res["abs_median_error"],
            "success": fit_res["success"],
            "weights": fit_res["weights"],
            "z_star": fit_res["z_star"],
            "start_id": fit_res["start_id"],
            "is_proxy_start": fit_res["is_proxy_start"],
            "objective_value": fit_res["objective_value"],
            "message": fit_res["message"],
            "all_start_results": fit_res["all_start_results"]
        })

    feasible = [d for d in lambda_results if d["abs_median_error"] <= median_tolerance]

    if feasible:
        chosen = sorted(
            feasible,
            key=lambda d: (d["target_mad"], d["mad"], d["lambda"])
        )[0]
    else:
        chosen = sorted(
            lambda_results,
            key=lambda d: (
                d["abs_median_error"],
                d["target_mad"],
                d["mad"],
                d["lambda"]
            )
        )[0]

    return chosen, lambda_results, proxy_w, proxy_obj

# ------------------------------------------------------------
# Check local Drive cache first, then GitHub
# ------------------------------------------------------------
for filename, local_path in FILE_MAP.items():
    ensure_file_from_github(filename, local_path)

cache_available = all(path.exists() for path in FILE_MAP.values())

if cache_available:
    print("\nLoading cached results ...")

    weights_df = pd.read_csv(WEIGHTS_FILE, index_col=0)
    frontier_train_true = pd.read_csv(FRONTIER_FILE)
    multistart_log_df = pd.read_csv(DIAGNOSTICS_FILE)
    lambda_insample_df = pd.read_csv(LAMBDA_FILE)
    comparison_df = pd.read_csv(COMPARISON_FILE)
    proxy_summary = pd.read_csv(PROXY_FILE)

    results_train_true = []
    for _, row in frontier_train_true.iterrows():
        col_name = f"target_{row['target_median']:.6f}"
        if col_name not in weights_df.columns:
            raise KeyError(f"Missing weight column {col_name} in cached weights file.")

        results_train_true.append({
            "weights": weights_df[col_name].values,
            "z_star": None,
            "median": row["median"],
            "mad": row["mad"],
            "target_mad": row["target_mad"],
            "median_error": row["median_error"],
            "target_median": row["target_median"],
            "chosen_lambda": row["chosen_lambda"],
            "objective_value": np.nan,
            "success": True,
            "message": "loaded from cache",
            "best_start_id": row["best_start_id"],
            "best_is_proxy_start": row["best_is_proxy_start"]
        })

    proxy_results = []
    for _, row in proxy_summary.iterrows():
        target = row["target_median"]
        proxy_w, proxy_obj, proxy_status = min_abs_dev_around_target(R_full, target)

        if proxy_w is None or proxy_status not in ["optimal", "optimal_inaccurate"]:
            raise RuntimeError(f"Failed to reconstruct proxy weights for target {target:.6f}")

        proxy_w = np.maximum(proxy_w, 0.0)
        proxy_w = proxy_w / proxy_w.sum()

        proxy_results.append({
            "target_median": target,
            "proxy_obj": proxy_obj,
            "proxy_true_median": row["proxy_true_median"],
            "proxy_true_mad": row["proxy_true_mad"],
            "proxy_target_mad": row["proxy_target_mad"],
            "proxy_median_error": row["proxy_median_error"],
            "proxy_abs_median_error": row["proxy_abs_median_error"],
            "proxy_weights": proxy_w
        })

else:
    print("\nNo cached results found. Running optimization ...")

    results_train_true = []
    proxy_results = []
    all_multistart_logs = []
    lambda_insample_logs = []

    for i, target in enumerate(target_medians):
        print(f"\nTarget {i+1}/10 — target median: {target:.6f}")

        chosen_res, lambda_results, proxy_w, proxy_obj = choose_lambda_for_target(
            R=R_full,
            target_median=target,
            lambda_grid=lambda_grid,
            median_tolerance=median_tolerance,
            n_starts=20,
            perturb_scale=2.5,
            maxiter=20000,
            seed=1000 + i
        )

        chosen_lambda = chosen_res["lambda"]

        for d in lambda_results:
            lambda_insample_logs.append({
                "target_median": target,
                "lambda": d["lambda"],
                "median": d["median"],
                "mad": d["mad"],
                "target_mad": d["target_mad"],
                "median_error": d["median_error"],
                "abs_median_error": d["abs_median_error"],
                "is_chosen_lambda": (d["lambda"] == chosen_lambda)
            })

        proxy_stats = compute_portfolio_stats_from_weights(R_full, proxy_w, target_median=target)

        proxy_results.append({
            "target_median": target,
            "proxy_obj": proxy_obj,
            "proxy_true_median": proxy_stats["median"],
            "proxy_true_mad": proxy_stats["mad"],
            "proxy_target_mad": proxy_stats["target_mad"],
            "proxy_median_error": proxy_stats["median"] - target,
            "proxy_abs_median_error": abs(proxy_stats["median"] - target),
            "proxy_weights": proxy_w
        })

        for d in chosen_res["all_start_results"]:
            all_multistart_logs.append({
                "target_median": target,
                "chosen_lambda": chosen_lambda,
                "start_id": d["start_id"],
                "is_proxy_start": d["is_proxy_start"],
                "median": d["median"],
                "mad": d["mad"],
                "target_mad": d["target_mad"],
                "median_error": d["median_error"],
                "abs_median_error": d["abs_median_error"],
                "objective_value": d["objective_value"],
                "success": d["success"]
            })

        results_train_true.append({
            "weights": chosen_res["weights"],
            "z_star": chosen_res["z_star"],
            "median": chosen_res["median"],
            "mad": chosen_res["mad"],
            "target_mad": chosen_res["target_mad"],
            "median_error": chosen_res["median_error"],
            "target_median": target,
            "chosen_lambda": chosen_lambda,
            "objective_value": chosen_res["objective_value"],
            "success": chosen_res["success"],
            "message": chosen_res["message"],
            "best_start_id": chosen_res["start_id"],
            "best_is_proxy_start": chosen_res["is_proxy_start"]
        })

        print(f"lambda = {chosen_lambda}")
        print(f"median = {chosen_res['median']:.12f}")
        print(f"MAD = {chosen_res['mad']:.6f}")
        print(f"TargetMAD = {chosen_res['target_mad']:.6f}")

    frontier_train_true = pd.DataFrame([
        {
            "target_median": r["target_median"],
            "median": r["median"],
            "mad": r["mad"],
            "target_mad": r["target_mad"],
            "median_error": r["median_error"],
            "abs_median_error": abs(r["median_error"]),
            "chosen_lambda": r["chosen_lambda"],
            "best_start_id": r["best_start_id"],
            "best_is_proxy_start": r["best_is_proxy_start"]
        }
        for r in results_train_true
    ]).sort_values("target_median").reset_index(drop=True)

    proxy_summary = pd.DataFrame([
        {
            "target_median": r["target_median"],
            "proxy_true_median": r["proxy_true_median"],
            "proxy_true_mad": r["proxy_true_mad"],
            "proxy_target_mad": r["proxy_target_mad"],
            "proxy_median_error": r["proxy_median_error"],
            "proxy_abs_median_error": r["proxy_abs_median_error"]
        }
        for r in proxy_results
    ]).sort_values("target_median").reset_index(drop=True)

    multistart_log_df = pd.DataFrame(all_multistart_logs).sort_values(
        ["target_median", "start_id"]
    ).reset_index(drop=True)

    lambda_insample_df = pd.DataFrame(lambda_insample_logs).sort_values(
        ["target_median", "lambda"]
    ).reset_index(drop=True)

    comparison_df = proxy_summary.merge(
        frontier_train_true,
        on="target_median",
        how="inner",
        suffixes=("_proxy", "_true")
    )

    print("\nFinal training frontier:")
    print(frontier_train_true[[
        "target_median",
        "median",
        "mad",
        "target_mad",
        "abs_median_error",
        "chosen_lambda"
    ]])

    weights_df = pd.DataFrame(
        {f"target_{r['target_median']:.6f}": r["weights"] for r in results_train_true},
        index=returns_train.columns
    )
    weights_df.to_csv(WEIGHTS_FILE)

    frontier_train_true.to_csv(FRONTIER_FILE, index=False)
    multistart_log_df.to_csv(DIAGNOSTICS_FILE, index=False)
    lambda_insample_df.to_csv(LAMBDA_FILE, index=False)
    comparison_df.to_csv(COMPARISON_FILE, index=False)
    proxy_summary.to_csv(PROXY_FILE, index=False)

    print("\nSaved results to:")
    print(f"  {WEIGHTS_FILE}")
    print(f"  {FRONTIER_FILE}")
    print(f"  {DIAGNOSTICS_FILE}")
    print(f"  {LAMBDA_FILE}")
    print(f"  {COMPARISON_FILE}")
    print(f"  {PROXY_FILE}")

The final training results show that the true-method portfolios are able to match almost exactly the target medians. This means that the penalty mechanism works well in-sample.

However, the value of λ changes a lot across target medians. This confirms that there is not a single penalty parameter that works equally well for all portfolios. Some target medians are easier to reach, while others require a much stronger penalty.

The MAD has a clear U-shape. It is very high for the most negative target median and also increases for the largest target medians, while it is lowest around the central target median, approximately $0.0008$. A possible interpretation is that, asking for very extreme medians forces the portfolio to become less diversified or more concentrated on particular stocks, and this increases the dispersion of returns.

In the next step we graphically compare the portfolios obtained with the proxy method and with the multi-start true method.

In [ ]:
# ============================================================
# 6) Plot the 10 median-MAD combinations
#    Proxy vs single-start vs multi-start
# ============================================================

import matplotlib.pyplot as plt

def compute_true_stats(R, weights):
    port_returns = R @ weights
    true_median = np.median(port_returns)
    true_mad = np.mean(np.abs(port_returns - true_median))
    return true_median, true_mad

R_train = returns_train.values
rows = []

n_portfolios = min(
    len(results_train_proxy),
    len(results_train_true)
)

for i in range(n_portfolios):
    proxy_res = results_train_proxy[i]
    multi_res = results_train_true[i]

    target = proxy_res["target_median"]

    proxy_med, proxy_mad = compute_true_stats(R_train, proxy_res["weights"])
    multi_med, multi_mad = compute_true_stats(R_train, multi_res["weights"])

    row = {
        "target_median": target,
        "proxy_true_median": proxy_med,
        "proxy_true_mad": proxy_mad,
        "multistart_true_median": multi_med,
        "multistart_true_mad": multi_mad,
    }

    if "results_train_single_start" in globals():
        single_res = results_train_single_start[i]
        single_med, single_mad = compute_true_stats(R_train, single_res["weights"])
        row["single_true_median"] = single_med
        row["single_true_mad"] = single_mad

    rows.append(row)

comparison_df = pd.DataFrame(rows).sort_values("target_median").reset_index(drop=True)

# ------------------------------------------------------------
# Plot 1: target median vs realized median
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

plt.plot(
    comparison_df["target_median"],
    comparison_df["proxy_true_median"],
    marker="o",
    label="Proxy method"
)

if "single_true_median" in comparison_df.columns:
    plt.plot(
        comparison_df["target_median"],
        comparison_df["single_true_median"],
        marker="o",
        label="Single-start true method"
    )

plt.plot(
    comparison_df["target_median"],
    comparison_df["multistart_true_median"],
    marker="o",
    label="Best multi-start method"
)

xmin = comparison_df["target_median"].min()
xmax = comparison_df["target_median"].max()
plt.plot([xmin, xmax], [xmin, xmax], linestyle="--", label="Perfect match (y=x)")

plt.xlabel("Target median")
plt.ylabel("True realized median")
plt.title("Target vs realized median")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Plot 2: median-MAD frontier
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

plt.plot(
    comparison_df["proxy_true_median"],
    comparison_df["proxy_true_mad"],
    marker="o",
    label="Proxy method"
)

if "single_true_median" in comparison_df.columns:
    plt.plot(
        comparison_df["single_true_median"],
        comparison_df["single_true_mad"],
        marker="o",
        label="Single-start true method"
    )

plt.plot(
    comparison_df["multistart_true_median"],
    comparison_df["multistart_true_mad"],
    marker="o",
    label="Best multi-start method"
)

plt.xlabel("True realized median")
plt.ylabel("True MAD")
plt.title("Median-MAD combinations")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

The first plot clearly shows the difference between the two methods. The best multi-start method is almost exactly on the dashed line, meaning that it reaches the target median basically perfectly.

The second plot shows that the multi-start method is not only better in matching the target median, but it also often gives a lower true MAD, especially at the extreme medians. This is an important result, because the proxy method looked computationally simpler, but it was not optimizing the true quantity we cared about.

In this final step we evaluate the portfolios out-of-sample. We take the weights obtained from the training sample and apply them to the test sample. For each portfolio we compute again the realized median, the true MAD, the TargetMAD and the median error.

This allows us to check whether the portfolios that worked well in-sample are also stable in the test period.

In [ ]:
# ============================================================
# Step 7: Evaluate train portfolios on the test sample
# ============================================================

# Test matrix
R_test = returns_test.values

def compute_out_of_sample_stats(R, weights, target_median):
    port_returns = R @ weights
    realized_median = np.median(port_returns)
    true_mad = np.mean(np.abs(port_returns - realized_median))
    target_mad = np.mean(np.abs(port_returns - target_median))
    median_error = realized_median - target_median

    return {
        "median": realized_median,
        "mad": true_mad,
        "target_mad": target_mad,
        "median_error": median_error,
        "abs_median_error": abs(median_error)
    }

# ------------------------------------------------------------
# Evaluate proxy and true-method portfolios on test sample
# ------------------------------------------------------------
proxy_test_results = []
true_test_results = []

n_portfolios = min(len(proxy_results), len(results_train_true))

for i in range(n_portfolios):
    proxy_res = proxy_results[i]
    true_res  = results_train_true[i]

    target = proxy_res["target_median"]

    proxy_eval = compute_out_of_sample_stats(
        R=R_test,
        weights=proxy_res["proxy_weights"],
        target_median=target
    )

    true_eval = compute_out_of_sample_stats(
        R=R_test,
        weights=true_res["weights"],
        target_median=target
    )

    proxy_test_results.append({
        "target_median": target,
        "median": proxy_eval["median"],
        "mad": proxy_eval["mad"],
        "target_mad": proxy_eval["target_mad"],
        "median_error": proxy_eval["median_error"],
        "abs_median_error": proxy_eval["abs_median_error"]
    })

    true_test_results.append({
        "target_median": target,
        "median": true_eval["median"],
        "mad": true_eval["mad"],
        "target_mad": true_eval["target_mad"],
        "median_error": true_eval["median_error"],
        "abs_median_error": true_eval["abs_median_error"]
    })

# ------------------------------------------------------------
# Convert to DataFrames
# ------------------------------------------------------------
proxy_test_df = pd.DataFrame(proxy_test_results).sort_values("target_median").reset_index(drop=True)
true_test_df  = pd.DataFrame(true_test_results).sort_values("target_median").reset_index(drop=True)

print("\nProxy portfolios evaluated on test sample:")
print(proxy_test_df)

print("\nTrue-method portfolios evaluated on test sample:")
print(true_test_df)

# ------------------------------------------------------------
# Also build train summary tables for plotting
# ------------------------------------------------------------
proxy_train_df = pd.DataFrame([
    {
        "target_median": r["target_median"],
        "median": r["proxy_true_median"],
        "mad": r["proxy_true_mad"],
        "target_mad": r["proxy_target_mad"],
        "median_error": r["proxy_median_error"],
        "abs_median_error": r["proxy_abs_median_error"]
    }
    for r in proxy_results
]).sort_values("target_median").reset_index(drop=True)

true_train_df = pd.DataFrame([
    {
        "target_median": r["target_median"],
        "median": r["median"],
        "mad": r["mad"],
        "target_mad": r["target_mad"],
        "median_error": r["median_error"],
        "abs_median_error": abs(r["median_error"])
    }
    for r in results_train_true
]).sort_values("target_median").reset_index(drop=True)

# ------------------------------------------------------------
# Plot 1: Step-6 / Step-7 style plot in the (median, MAD) plane
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

plt.plot(
    proxy_train_df["median"], proxy_train_df["mad"],
    marker="o", linestyle="-", label="Proxy train"
)

plt.plot(
    proxy_test_df["median"], proxy_test_df["mad"],
    marker="o", linestyle="--", label="Proxy test"
)

plt.plot(
    true_train_df["median"], true_train_df["mad"],
    marker="o", linestyle="-", label="True-method train"
)

plt.plot(
    true_test_df["median"], true_test_df["mad"],
    marker="o", linestyle="--", label="True-method test"
)

plt.xlabel("Realized median")
plt.ylabel("MAD")
plt.title("Step 6 and Step 7: train and test median-MAD combinations")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Plot 2: Homogeneous comparison in the (median error, TargetMAD) logic
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

plt.plot(
    proxy_train_df["target_median"], proxy_train_df["target_mad"],
    marker="o", linestyle="-", label="Proxy train TargetMAD"
)

plt.plot(
    proxy_test_df["target_median"], proxy_test_df["target_mad"],
    marker="o", linestyle="--", label="Proxy test TargetMAD"
)

plt.plot(
    true_train_df["target_median"], true_train_df["target_mad"],
    marker="o", linestyle="-", label="True-method train TargetMAD"
)

plt.plot(
    true_test_df["target_median"], true_test_df["target_mad"],
    marker="o", linestyle="--", label="True-method test TargetMAD"
)

plt.xlabel("Target median")
plt.ylabel("TargetMAD")
plt.title("Train and test TargetMAD by target median")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Save outputs
# ------------------------------------------------------------
proxy_test_df.to_csv("proxy_test_summary.csv", index=False)
true_test_df.to_csv("true_method_test_summary.csv", index=False)

print("\nSaved:")
print("  proxy_test_summary.csv")
print("  true_method_test_summary.csv")

The out-of-sample results are much weaker than the in-sample ones. For the proxy portfolios, the test medians are all concentrated around $0.0009$ and $0.0010$, almost independently from the original target median.

The second method portfolios also lose part of their in-sample precision. In training they matched the target medians almost exactly, but in the test sample the realized medians can be quite far from the targets, especially for the extreme portfolios. This suggests that the true-method portfolios are more exposed to overfitting: they satisfy the median constraint very well in-sample, but this constraint is not stable out-of-sample.

The first test plot confirms this interpretation. The training curve of the true method has the expected U-shape, while the test curve is much more irregular.

The second test plot is even clearer. The proxy test TargetMAD is almost flat and close to $0.005$, while the true-method test TargetMAD is larger at the extremes and more unstable. Therefore, the true method is better in-sample and better for exactly satisfying the optimization constraint, but the proxy method appears more robust out-of-sample.

## Interaction with AI and evaluation of its contribution

Throughout the assignment, we used AI systems to generate code, debug errors, and explore alternative methodologies.

In the first part, AI was useful for building the data pipeline quickly. It suggested using `yfinance` together with a public GitHub repository containing the S&P 500 constituent list, and helped structure the code for downloading, cleaning, and storing the data. AI also contributed to the descriptive statistics, PCA, clustering, and autoencoder sections, in some cases introducing methods we were not initially familiar with, such as the Lilliefors test, the Davies-Bouldin index, and alternative hierarchical clustering linkages.

The optimization stage was more problematic. Minimizing portfolio MAD subject to a target median is non-trivial because the portfolio median is not smooth in the weights. AI initially proposed approximation-based approaches that did not correctly solve the intended problem, in several cases the realized medians were far from the targets. We therefore could not simply accept the generated code, and had to verify it carefully. When uncertain about specific implementations, we sometimes asked a separate AI system to explain the same code independently, without revealing our interpretation, to check whether it was actually doing what we intended.

AI was also useful in the autoencoder section for diagnosing instability and convergence problems, describing the loss and gradient behavior to the AI helped narrow down potential causes and hyperparameter choices.

That said, we observed several recurring limitations:

- AI often produces solutions that look correct but rely on hidden approximations or wrong assumptions. For example, in the portfolio optimization section, AI initially suggested replacing the true portfolio median with a weighted average of individual stock medians. This made the problem linear and easy to solve, but the realized portfolio medians were not always equal to the desired target medians.

- AI can ignore relevant statistical prerequisites. For instance, when we first plotted the distribution of average daily market returns, AI suggested explaining the apparent normality through the Central Limit Theorem. However, this argument is not fully appropriate because daily stock returns are unlikely to be independent and identically distributed.

- Optimization code generated by AI may be numerically unstable or solve a different problem than intended. In our case, some early optimization attempts minimized deviations from an approximated median rather than the true portfolio median. Later, when we tried to directly penalize deviations from the target median, the optimization often reached the maximum number of iterations before convergence.

- AI tends to sound confident even when the answer is uncertain. For example, when we asked about the unusually low rolling correlation between PC1 and stock returns around November 2017, AI proposed an explanation based on a rotation from technology to financial stocks. We then checked the rolling correlation between the two sectors and found that this explanation was only partially convincing.


Overall, AI is most useful as an assistant for exploration and implementation. Verification and domain knowledge remain essential, especially in finance where small modeling errors can have large effects on results.

## Final Comment

The assignment confirmed several known difficulties in applying machine learning to financial data.

Descriptive analysis showed the usual stylized facts: fat tails, non-normality, and strong market components in PCA. The clustering and autoencoder sections showed how unstable financial data can be: small methodological changes often produced very different structures, suggesting the signal is weak relative to the noise.

The optimization section was the most technically demanding part. Median-based portfolio optimization is harder than mean-variance because the portfolio median is not differentiable in the weights. Exact methods achieved very accurate target medians in-sample but degraded more out-of-sample. The proxy method was less precise in-sample but more stable on the test data. This is consistent with a general pattern: in noisy, limited-sample settings, simpler and more robust methods often generalize better than highly optimized ones.

More generally, financial return series are noisy, non-stationary, and driven by factors that change over time. This limits how much even sophisticated methods can extract in terms of stable, meaningful patterns.

On the AI side, the main takeaway is that AI-generated code must always be checked. Several times AI proposed methods that looked reasonable but were theoretically wrong, relied on bad approximations, or produced unstable results. Domain knowledge and careful validation are not optional when using AI in quantitative finance.

# Question 2 — Learning Option Delta with Machine Learning & Deep Learning

**Course:** 21046 Data Science and Machine Learning for Finance (Bocconi University, 2025/2026)

---

> **AI-contribution disclosure (global, per-cell detail in Sec. 7.6).** The
> overall notebook structure, boilerplate code and initial model implementations
> were drafted with the assistance of an AI coding assistant (GitHub Copilot /
> Colab AI). Cells where AI scaffolding was used carry a brief inline comment
> starting with `# AI-assistance note:`. All financial interpretations, modelling
> choices, hyperparameter decisions, and critical analysis are the authors' own
> contribution; see Sec. 7.6 for the full breakdown.

> **Note on benchmarks.** Two analytical Black 76 benchmarks are reported
> throughout: one driven by **rolling realised volatility** ("RV-Black")
> and one driven by **option-specific implied volatility back-solved from the
> quote** ("IV-Black"). The IV-Black benchmark is an upper bound on what
> any model can achieve when supervised on the vendor delta because IV is the
> exact non-linear summary that the market is using to set delta. The
> ML/DL contribution is therefore framed as **smile/skew correction on top
> of RV-Black**, not as a replacement of the Black formula.

In [ ]:
# == Section 0: Setup & Imports ==================================================
# AI-assistance note: imports list and plot-style block were initially scaffolded
# with Colab AI; library choices (shap, scipy.optimize.brentq, keras) and every
# financial-modelling step below are the authors' own work.
# Some packages are not pre-installed on vanilla Colab kernels.
try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'shap'])
    import shap

import warnings, os, random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (registers 3D projection)
import seaborn as sns

from scipy.stats import norm
from scipy.optimize import brentq
from datetime import datetime

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Used to render result tables as DataFrames (rather than as plain text via print).
from IPython.display import display

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# Plot style
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'font.size': 11,
    'legend.fontsize': 10,
})
sns.set_style('whitegrid')

print("All imports successful.")
print(f"  TensorFlow version: {tf.__version__}")
print(f"  SHAP version:       {shap.__version__}")


---
## Section 1 — Data Loading & Exploratory Data Analysis

We load the E-Mini S&P 500 options dataset. The data contains daily observations of:
- Continuous and Dec-2023 futures settlement prices.
- Call and Put delta, gamma, and price for 5 strikes: 4000, 4200, 4400, 4600, 4800.

All options and the Dec-2023 futures expire on **December 15, 2023**.

In [ ]:
# == 1.1 Load raw data =========================================================
# Reproducibility: try the local repo copy first, then fall back to the GitHub
# raw URL. Pinning a commit SHA in the URL keeps Colab runs reproducible even
# after the repo evolves.
import os

LOCAL_CSV  = os.path.join('..', 'data', 'takehome21046data2026.csv')
LOCAL_CSV2 = os.path.join('data', 'takehome21046data2026.csv')
REMOTE_CSV = (
    'https://raw.githubusercontent.com/stfgrz/21046-mlf-ps/'
    'c2ea2967fcdead9179826f98a46758e8fb703253/data/takehome21046data2026.csv'
)

if os.path.exists(LOCAL_CSV):
    raw = pd.read_csv(LOCAL_CSV)
    print(f"Loaded local copy: {LOCAL_CSV}")
elif os.path.exists(LOCAL_CSV2):
    raw = pd.read_csv(LOCAL_CSV2)
    print(f"Loaded local copy: {LOCAL_CSV2}")
else:
    raw = pd.read_csv(REMOTE_CSV)
    print(f"Loaded remote copy: {REMOTE_CSV}")
print(f"Shape: {raw.shape}")
raw.head(3)

In [ ]:
# == 1.2 Rename columns for convenience ========================================
col_map = {}
for c in raw.columns:
    cl = c.strip()
    if cl == 'Date':
        col_map[c] = 'date'
    elif cl == 'Date.1':
        col_map[c] = 'date_dup'
    elif 'CONT' in cl:
        col_map[c] = 'F_cont'
    elif 'DEC 2023 - SETT' in cl:
        col_map[c] = 'F_dec23'
    else:
        parts = cl.replace(' - ', '_').split()
        opt_type = parts[0].lower()
        strike   = parts[3].split('_')[0]
        if 'DELTA' in cl:
            metric = 'delta'
        elif 'GAMMA' in cl:
            metric = 'gamma'
        else:
            metric = 'price'
        col_map[c] = f"{opt_type}_{metric}_{strike}"

raw.rename(columns=col_map, inplace=True)
raw.drop(columns=['date_dup'], inplace=True)
raw['date'] = pd.to_datetime(raw['date'])
raw.set_index('date', inplace=True)
raw = raw.sort_index()

print(f"Columns ({len(raw.columns)}):")
for i, c in enumerate(raw.columns):
    print(f"  {i:2d}. {c}")

# --- Data-quality snapshot --------------------------------------------------
# We surface basic sample size, date range, and per-column NaN counts so any
# downstream filtering (NaN volatility, NaN IV) can be put in context.
n_rows, n_cols = raw.shape
print(f"\nSample window : {raw.index.min().date()}  ->  {raw.index.max().date()} "
      f"({n_rows} business days, {n_cols} columns)")

nan_counts = raw.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts):
    print("\nColumns with missing values (raw panel):")
    for col, cnt in nan_counts.items():
        print(f"  {col:25s}  {cnt:>4d} NaN  ({100 * cnt / n_rows:5.2f} %)")
else:
    print("\nNo missing values detected in the raw panel.")


In [ ]:
# == 1.3 Quick EDA — futures prices ============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

raw['F_cont'].plot(ax=axes[0], color='steelblue', lw=1.2)
axes[0].set_title('E-Mini S&P 500 Continuous Futures')
axes[0].set_ylabel('Settlement Price')

raw['F_dec23'].plot(ax=axes[1], color='darkorange', lw=1.2)
axes[1].set_title('E-Mini S&P 500 Dec 2023 Contract')
axes[1].set_ylabel('Settlement Price')

plt.tight_layout()
plt.show()

In [ ]:
# == 1.4 Reshape to long panel =================================================
strikes = [4000, 4200, 4400, 4600, 4800]
records = []

for dt, row in raw.iterrows():
    for K in strikes:
        records.append({
            'date':       dt,
            'F_cont':     row['F_cont'],
            'F_dec23':    row['F_dec23'],
            'strike':     K,
            'call_delta': row[f'call_delta_{K}'],
            'call_gamma': row[f'call_gamma_{K}'],
            'call_price': row[f'call_price_{K}'],
            'put_delta':  row[f'put_delta_{K}'],
            'put_gamma':  row[f'put_gamma_{K}'],
            'put_price':  row[f'put_price_{K}'],
        })

df = pd.DataFrame(records)
print(f"Long panel shape: {df.shape}")
df.head()

In [ ]:
# == 1.5 Heatmap: call delta across strikes & time ============================
pivot = df.pivot_table(index='date', columns='strike', values='call_delta')
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot.T, cmap='RdYlGn', ax=ax, cbar_kws={'label': 'Call Delta'})
ax.set_title('Call Delta Heatmap (Strike x Date)')
ax.set_ylabel('Strike')
ax.set_xlabel('Date')
xticks = ax.get_xticks()
ax.set_xticks(xticks[::60])
ax.set_xticklabels([pivot.index[int(i)].strftime('%Y-%m')
                     for i in xticks[::60] if int(i) < len(pivot)], rotation=45)
plt.tight_layout()
plt.show()

---
## Section 2 — Feature Engineering

We construct the features that the question prescribes:
1. **Underlying price** $F$ — the Dec-2023 futures settlement price.
2. **Realised volatility** $\sigma$ — rolling 21-day standard deviation of daily log-returns on the *continuous* futures series, annualised ($\times\sqrt{252}$).
3. **Strike** $K$ — discrete values 4000, 4200, 4400, 4600, 4800.
4. **Time to maturity** $\tau$ — business days from observation date to expiry (Dec 15, 2023), expressed in years ($\div 252$).
5. **Moneyness** $m = F/K$ and **log-moneyness** $\ln(F/K)$ — standard non-dimensionalised representations used in options pricing (see Hull, *Options, Futures, and Other Derivatives*, Ch. 19).

In [ ]:
# == 2.1 Time to maturity =====================================================
EXPIRY = pd.Timestamp('2023-12-15')

df['T_days'] = np.busday_count(
    df['date'].values.astype('datetime64[D]'),
    np.datetime64(EXPIRY, 'D')
)
df['T'] = df['T_days'] / 252.0

# Drop rows where T <= 0 (at or past expiry)
df = df[df['T'] > 0].copy()
print(f"After dropping T<=0: {len(df)} rows")
print(f"T range: {df['T'].min():.4f} to {df['T'].max():.4f} years")

In [ ]:
# == 2.2 Realised volatility (rolling 21-day on continuous futures) ============
ret = raw['F_cont'].pct_change().apply(np.log1p)
roll_vol = ret.rolling(21).std() * np.sqrt(252)
roll_vol.name = 'sigma'

df = df.merge(roll_vol, left_on='date', right_index=True, how='left')
df.dropna(subset=['sigma'], inplace=True)
print(f"After dropping NaN vol: {len(df)} rows")
print(f"sigma range: {df['sigma'].min():.4f} to {df['sigma'].max():.4f}")

In [ ]:
# == 2.3 Moneyness & log-moneyness ============================================
df['moneyness']     = df['F_dec23'] / df['strike']
df['log_moneyness'] = np.log(df['moneyness'])

print(df[['date','strike','F_dec23','moneyness','log_moneyness','T','sigma']].head(10))

In [ ]:
# == 2.4 Feature summary statistics ============================================
feature_cols = ['moneyness', 'log_moneyness', 'T', 'sigma']
print(df[feature_cols + ['call_delta']].describe().round(4))

---
## Section 3 — Black 76 Analytical Delta (Benchmark)

### Financial Background

Since the underlying is a **futures** contract, the appropriate model is the **Black 76 model** (Black, 1976), not standard Black–Scholes. The futures price tracks the forward, not the spot, so under the risk-neutral measure $F$ is a martingale and the call/put price formulas are

$$C = e^{-rT}\bigl[F\Phi(d_1) - K\Phi(d_2)\bigr], \qquad P = e^{-rT}\bigl[K\Phi(-d_2) - F\Phi(-d_1)\bigr],$$

with $d_1 = \bigl[\ln(F/K) + \tfrac{1}{2}\sigma^2 T\bigr]/(\sigma\sqrt{T})$ and $d_2 = d_1 - \sigma\sqrt{T}$.

### Two delta conventions

For options on futures there are **two** common deltas:
- **Premium-paid (PP):** $\Delta^{\text{PP}}_{\text{call}} = e^{-rT}\Phi(d_1)$, $\Delta^{\text{PP}}_{\text{put}} = -e^{-rT}\Phi(-d_1)$. Used by exchanges that *settle* the option premium up-front (the European/Black 76 textbook delta).
- **Futures-style / undiscounted (FS):** $\Delta^{\text{FS}}_{\text{call}} = \Phi(d_1)$, $\Delta^{\text{FS}}_{\text{put}} = \Phi(d_1) - 1$. Used by CME-style margined options where the premium itself is mark-to-market.

The two differ by the discount factor $e^{-rT}$ and produce *different* hedge ratios. Which convention the data is using cannot be guessed from realised-vol MAE — that is a noisy diagnostic that mixes convention error with the volatility-proxy error. The correct test is **put-call parity on deltas**, which holds *exactly* in each convention regardless of $\sigma$:

$$\Delta^{\text{PP}}_{\text{call}} - \Delta^{\text{PP}}_{\text{put}} = e^{-rT}, \qquad \Delta^{\text{FS}}_{\text{call}} - \Delta^{\text{FS}}_{\text{put}} = 1.$$

We therefore choose the convention from the empirical parity of the *quoted deltas*, not from MAE against an RV-driven Black formula.

### Two analytical baselines (RV-Black and IV-Black)

We will build **two** Black 76 baselines:
1. **RV-Black** — the formula fed with a rolling 21-day realised vol $\sigma^{\text{real}}$, identical across strikes on a given day.
2. **IV-Black** — the formula fed with each option's own implied vol $\sigma^{\text{iv}}$, back-solved from the quoted price (Sec. 3.3).

The IV-Black delta is essentially the *target* function the market is using; it serves as an upper bound on attainable accuracy when supervising on the vendor delta. The RV-Black delta is the structurally honest baseline a hedger would build *without* a vendor IV surface, which is the harder benchmark for ML/DL to beat.

### References
- Black, F. (1976). The pricing of commodity contracts. *Journal of Financial Economics*, 3(1–2), 167–179.
- Hull, J. C. (2022). *Options, Futures, and Other Derivatives* (11th ed.), Pearson, Ch. 18–19.

In [ ]:
# == 3.1 Black 76 delta & convention selection (parity-driven) ================
# AI-assistance note: the formula scaffolding (functions for d1, prices, deltas)
# was first drafted with AI help. The convention-selection logic is now driven
# by the put-call parity constraint instead of by realised-vol MAE — the
# latter is noisy because it mixes the convention error with the
# realised-vol-vs-implied-vol bias. See Sec. 3 markdown for the derivation.
R_F = 0.053  # risk-free rate (annualised) — calibration discussed in the markdown

def _d1(F, K, T, sigma):
    return (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * np.sqrt(T))

# --- Call / Put DELTA: two conventions --------------------------------------
def black76_call_delta_pp(F, K, T, sigma, r=R_F):
    '''Premium-paid (European Black 76): e^{-rT} * N(d1).'''
    return np.exp(-r * T) * norm.cdf(_d1(F, K, T, sigma))

def black76_call_delta_fs(F, K, T, sigma):
    '''Futures-style / undiscounted (CME-style quoting): N(d1).'''
    return norm.cdf(_d1(F, K, T, sigma))

def black76_put_delta_pp(F, K, T, sigma, r=R_F):
    '''Premium-paid put delta: -e^{-rT} * N(-d1).'''
    return -np.exp(-r * T) * norm.cdf(-_d1(F, K, T, sigma))

def black76_put_delta_fs(F, K, T, sigma):
    '''Futures-style put delta: N(d1) - 1.'''
    return norm.cdf(_d1(F, K, T, sigma)) - 1.0

# --- Black 76 PRICE (needed for IV inversion and P&L tests) -----------------
def black76_call_price(F, K, T, sigma, r=R_F):
    d1 = _d1(F, K, T, sigma); d2 = d1 - sigma * np.sqrt(T)
    return np.exp(-r * T) * (F * norm.cdf(d1) - K * norm.cdf(d2))

def black76_put_price(F, K, T, sigma, r=R_F):
    d1 = _d1(F, K, T, sigma); d2 = d1 - sigma * np.sqrt(T)
    return np.exp(-r * T) * (K * norm.cdf(-d2) - F * norm.cdf(-d1))

# --- Compute both conventions on the panel (RV-Black) -----------------------
df['black_rv_call_pp'] = black76_call_delta_pp(df['F_dec23'], df['strike'], df['T'], df['sigma'])
df['black_rv_call_fs'] = black76_call_delta_fs(df['F_dec23'], df['strike'], df['T'], df['sigma'])
df['black_rv_put_pp']  = black76_put_delta_pp(df['F_dec23'], df['strike'], df['T'], df['sigma'])
df['black_rv_put_fs']  = black76_put_delta_fs(df['F_dec23'], df['strike'], df['T'], df['sigma'])

# --- Convention selection via put-call parity (vol-free) --------------------
parity_market = df['call_delta'] - df['put_delta']
mae_parity_pp = (parity_market - np.exp(-R_F * df['T'])).abs().mean()
mae_parity_fs = (parity_market - 1.0).abs().mean()
print("Put-call parity diagnostic (vol-free, drives convention choice):")
print(f"  Mean |c - p - e^(-rT)|  : {mae_parity_pp:.5f}   (premium-paid)")
print(f"  Mean |c - p - 1|        : {mae_parity_fs:.5f}   (futures-style)")

if mae_parity_fs <= mae_parity_pp:
    CHOSEN_CONVENTION = 'futures-style (N(d1))'
    df['black_delta']     = df['black_rv_call_fs']
    df['black_put_delta'] = df['black_rv_put_fs']
else:
    CHOSEN_CONVENTION = 'premium-paid (e^(-rT) N(d1))'
    df['black_delta']     = df['black_rv_call_pp']
    df['black_put_delta'] = df['black_rv_put_pp']
print(f"\n==> Reference Black 76 convention adopted (parity-driven): {CHOSEN_CONVENTION}")

# --- For information only: realised-vol MAE in both conventions ------------
mae_pp = mean_absolute_error(df['call_delta'], df['black_rv_call_pp'])
mae_fs = mean_absolute_error(df['call_delta'], df['black_rv_call_fs'])
print("\nRealised-vol Black 76 MAE on the full panel (informational only):")
print(f"  premium-paid  e^(-rT)*N(d1) : {mae_pp:.5f}")
print(f"  futures-style N(d1)         : {mae_fs:.5f}")
print("  (We do NOT use this MAE to pick the convention — it conflates the")
print("   convention choice with the realised-vs-implied vol bias.)")

print("\nBlack call delta summary (chosen convention):")
display(df['black_delta'].describe().round(4).to_frame())
print("Black put delta summary (chosen convention):")
display(df['black_put_delta'].describe().round(4).to_frame())

In [ ]:
# == 3.2 Compare RV-Black delta with market delta =============================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df['black_delta'], df['call_delta'], s=5, alpha=0.4, c='steelblue')
axes[0].plot([0, 1], [0, 1], 'r--', lw=1.5, label='45 degree line')
axes[0].set_xlabel(f'RV-Black 76 Delta ({CHOSEN_CONVENTION})')
axes[0].set_ylabel('Market Delta')
axes[0].set_title('Market vs RV-Black Delta')
axes[0].legend()

resid = df['call_delta'] - df['black_delta']
axes[1].hist(resid, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', ls='--', lw=1.5)
axes[1].set_xlabel('Market Delta minus RV-Black Delta')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residuals: Market minus RV-Black Delta')

plt.tight_layout()
plt.show()

mae_black  = mean_absolute_error(df['call_delta'], df['black_delta'])
rmse_black = np.sqrt(mean_squared_error(df['call_delta'], df['black_delta']))
r2_black   = r2_score(df['call_delta'], df['black_delta'])
print(f"RV-Black 76 vs Market -- MAE: {mae_black:.4f}  RMSE: {rmse_black:.4f}  R2: {r2_black:.4f}")

---
### 3.2bis — Risk-Free Rate Sensitivity

Our benchmark uses $r = 5.3\%$, calibrated to the average effective Federal Funds Rate
over 2022–2023 (FRED series `DFF`). The discount factor $e^{-rT}$ enters the *premium-paid*
delta convention but not the *futures-style* one, so the size of any rate-driven bias
depends on which convention the market data follows.

We re-evaluate the call-delta MAE at $r \in \{4\%, 5.3\%, 6\%\}$ to bracket the
plausible 1-year T-bill range over the sample period and confirm the comparison is
robust to a $\pm 70$ bp shift.

In [ ]:
# == 3.2bis Risk-free rate sensitivity (Black 76 vs market delta) ==============
sens_rows = []
for r_test in (0.040, 0.053, 0.060):
    pp = np.exp(-r_test * df['T']) * norm.cdf(_d1(df['F_dec23'], df['strike'],
                                                  df['T'], df['sigma']))
    fs = norm.cdf(_d1(df['F_dec23'], df['strike'], df['T'], df['sigma']))
    sens_rows.append({
        'r':                 r_test,
        'MAE premium-paid':  mean_absolute_error(df['call_delta'], pp),
        'MAE futures-style': mean_absolute_error(df['call_delta'], fs),
        'mean e^(-rT)':      float(np.exp(-r_test * df['T']).mean()),
    })

sens = pd.DataFrame(sens_rows).set_index('r').round(5)
print("RV-Black 76 call-delta MAE vs market under different risk-free rates:")
display(sens)

if CHOSEN_CONVENTION.startswith('futures'):
    print(f"\nThe chosen futures-style convention (N(d1)) is independent of r,")
    print(f"so the headline RV-Black MAE is invariant to the risk-free assumption.")
    print(f"r still enters the IV inversion (via discounting of the price), but")
    print(f"the price equation is monotone in sigma so the IV root is robust")
    print(f"under a +/-70 bp shift around our R_F = {R_F:.3f}.")
else:
    print(f"\nThe chosen premium-paid convention has an e^(-rT) factor; the table")
    print(f"shows the MAE only shifts at the 4th-5th decimal across the +/-70 bp")
    print(f"range, confirming our R_F = {R_F:.3f} choice does not drive the result.")

---
### 3.3 — Implied Volatility Extraction

A structurally relevant additional feature is the **Black 76 implied volatility** of each option, back-solved from the market price. Given $(F, K, T, C^{\text{mkt}})$ and a risk-free rate $r$, we define

$$\sigma^{\text{iv}} = \underset{\sigma > 0}{\text{solve}}\,\Big[\,C^{\text{Black76}}(F, K, T, \sigma, r) = C^{\text{mkt}}\,\Big]$$

and invert numerically using **Brent's method** (`scipy.optimize.brentq`) on the bracket $[10^{-4}, 5.0]$.

**Why add IV as a feature?**
1. It collapses the unobserved **volatility smile/skew** onto a single, explicit, strike-dependent input.
2. The model can then focus on residual non-linearities beyond Black 76 rather than re-learning the smile from scratch.
3. It is much closer to how trading desks actually calibrate hedge ratios in practice.

**Handling failures.** Rows where the Brent bracket has the same sign at both ends (deep-OTM noise, near-expiry floor effects, or arbitrage-violating quotes) are returned as `NaN`. We keep them in the main panel and simply drop them when training the IV-augmented variant of the best ML model (Section 4.7).

In [ ]:
# == 3.3 Implied volatility extraction + IV-Black analytical delta ============
# AI-assistance note: scaffolding for the brentq wrapper was suggested by Colab
# AI; bracket selection, NaN handling, and the strike-by-strike diagnostic were
# added by the authors after observing several failed inversions in deep-OTM
# regions.
def implied_vol_black76(price, F, K, T, r=R_F, option='call',
                        lower=1e-4, upper=5.0):
    '''Invert Black 76 to obtain implied vol. Returns NaN on failure.'''
    if not np.isfinite(price) or price <= 0 or T <= 0 or F <= 0 or K <= 0:
        return np.nan
    pricer = black76_call_price if option == 'call' else black76_put_price
    def fn(sigma):
        return pricer(F, K, T, sigma, r) - price
    try:
        f_lo, f_hi = fn(lower), fn(upper)
        if not (np.isfinite(f_lo) and np.isfinite(f_hi)) or f_lo * f_hi > 0:
            return np.nan
        return brentq(fn, lower, upper, maxiter=200, xtol=1e-7)
    except (ValueError, RuntimeError):
        return np.nan

print("Extracting call implied volatility ...")
df['iv_call'] = [
    implied_vol_black76(c, F, K, T, option='call')
    for c, F, K, T in zip(df['call_price'], df['F_dec23'], df['strike'], df['T'])
]
print("Extracting put implied volatility ...")
df['iv_put'] = [
    implied_vol_black76(p, F, K, T, option='put')
    for p, F, K, T in zip(df['put_price'], df['F_dec23'], df['strike'], df['T'])
]

n_call = int(df['iv_call'].notna().sum())
n_put  = int(df['iv_put'].notna().sum())
n_tot  = len(df)
print(f"IV (call) extracted: {n_call}/{n_tot}  ({100 * n_call / n_tot:5.2f} % success)")
print(f"IV (put)  extracted: {n_put}/{n_tot}  ({100 * n_put / n_tot:5.2f} % success)")
print(f"IV (call) median: {df['iv_call'].median():.4f}   |   "
      f"realised sigma median: {df['sigma'].median():.4f}")

# --- IV-Black analytical delta in BOTH conventions --------------------------
df['black_iv_call_pp'] = black76_call_delta_pp(df['F_dec23'], df['strike'], df['T'], df['iv_call'])
df['black_iv_call_fs'] = black76_call_delta_fs(df['F_dec23'], df['strike'], df['T'], df['iv_call'])
df['black_iv_put_pp']  = black76_put_delta_pp(df['F_dec23'], df['strike'], df['T'], df['iv_put'])
df['black_iv_put_fs']  = black76_put_delta_fs(df['F_dec23'], df['strike'], df['T'], df['iv_put'])

if CHOSEN_CONVENTION.startswith('futures'):
    df['black_iv_delta']     = df['black_iv_call_fs']
    df['black_iv_put_delta'] = df['black_iv_put_fs']
else:
    df['black_iv_delta']     = df['black_iv_call_pp']
    df['black_iv_put_delta'] = df['black_iv_put_pp']

# IV-Black panel-wide MAE — should be tiny because IV is back-solved from price
ok_c = df['iv_call'].notna()
ok_p = df['iv_put'].notna()
mae_iv_call = mean_absolute_error(df.loc[ok_c, 'call_delta'], df.loc[ok_c, 'black_iv_delta'])
mae_iv_put  = mean_absolute_error(df.loc[ok_p, 'put_delta'],  df.loc[ok_p, 'black_iv_put_delta'])
print("\nAnalytical IV-Black delta MAE on the full panel (chosen convention):")
print(f"  Call : {mae_iv_call:.5f}")
print(f"  Put  : {mae_iv_put:.5f}")
print("  These are the practical lower bounds for any model supervised on")
print("  the vendor delta; the residual is numerical (Brent tolerance, quote")
print("  rounding, settlement timing).")

# Strike-level success rate
iv_success = (df.assign(call_ok=df['iv_call'].notna(),
                         put_ok =df['iv_put'].notna())
                .groupby('strike')[['call_ok', 'put_ok']]
                .mean()
                .mul(100).round(2))
iv_success.columns = ['IV(call) success %', 'IV(put) success %']
print("\nIV inversion success rate by strike:")
display(iv_success)

# --- Plot the smile and the global IV cloud ---------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sample_dates = df['date'].unique()
pick = sample_dates[::max(1, len(sample_dates) // 8)][:8]
colors = plt.cm.viridis(np.linspace(0, 1, len(pick)))
for dt, col in zip(pick, colors):
    snap = df[df['date'] == dt].sort_values('moneyness')
    axes[0].plot(snap['moneyness'], snap['iv_call'],
                 marker='o', lw=1, color=col, alpha=0.9,
                 label=str(pd.Timestamp(dt).date()))
axes[0].set_xlabel('Moneyness $F/K$')
axes[0].set_ylabel('Implied Volatility')
axes[0].set_title('Call Implied Vol Smile across Sample Dates')
axes[0].legend(fontsize=8, ncol=2)

sc = axes[1].scatter(df['moneyness'], df['iv_call'],
                     s=5, alpha=0.35, c=df['T'], cmap='viridis')
plt.colorbar(sc, ax=axes[1], label=r'$\tau$ (years)')
axes[1].set_xlabel('Moneyness $F/K$')
axes[1].set_ylabel('Implied Volatility')
axes[1].set_title(r'Full-Sample IV vs Moneyness (colour = $\tau$)')

plt.tight_layout()
plt.show()

> **Interpretation.** Two analytical Black 76 deltas are now in scope:
> 1. **RV-Black** uses a rolling realised vol $\sigma^{\text{real}}$, identical
>    across strikes on each day. It is the structurally honest baseline a hedger
>    builds *without* a vendor IV surface and is non-trivial to beat: realised
>    vol misses the smile/skew but is not biased by self-supervised information.
> 2. **IV-Black** uses each option's own implied vol $\sigma^{\text{iv}}$
>    back-solved from the quote. Because $\sigma^{\text{iv}}$ is built from
>    the same equation that defines delta, IV-Black recovers the vendor delta
>    almost exactly (MAE ≈ $5\times10^{-3}$ on this panel). It is the *practical
>    lower bound* on attainable error when supervising on the quoted delta and
>    not a fair target for ML to beat.
>
> The ML/DL contribution is therefore best framed as a **smile/skew correction
> on top of RV-Black** — capturing the strike- and maturity-dependent residual
> that a single-vol Black formula cannot — rather than as a competitor to the
> IV-Black benchmark.

---
## Section 4 — Machine Learning Models

We train four ML models to learn the call delta as a function of the assignment-prescribed inputs $(F, K, \tau, \sigma)$ together with the standard non-dimensionalised representations $(F/K, \ln(F/K))$ that make the delta surface comparable across maturities:

1. **Ridge Regression** — $L_2$-regularised linear baseline.
2. **Lasso Regression** — $L_1$-regularised linear baseline (built-in feature selection).
3. **Random Forest** — non-parametric, captures non-linear interactions.
4. **Gradient Boosting** — gradient-boosted trees, typically the strongest non-linear baseline.

**Targets.** We train each model on two distinct targets:

- **Raw target** — the quoted market delta. This is the most direct supervised signal but, as we show in Sec. 6, the IV-Black analytical formula already recovers it almost perfectly so the ML upside is limited.
- **Residual target** — `market_delta - black_rv_delta`. This isolates the **smile/skew correction** that an RV-Black hedger is missing. Adding the RV-Black delta back at predict time gives the model's effective delta. This is the academically interesting framing because the residual is what ML can plausibly learn from $(F, K, \tau, \sigma)$.

**Train/test split:** 80% / 20% chronological split — preserves the temporal ordering, no data leakage from the future.

**Cross-validation:** **date-blocked K-fold** — we split the *unique trading dates* of the training window into 5 ordered blocks and include all five strikes per date in the same fold. A naive `TimeSeriesSplit` on row count would split a single date across train/validation because each date appears as five consecutive rows.

In [ ]:
# == 4.1 Prepare features, splits, residual targets, and date-blocked CV ======
# Features as prescribed by the assignment (F, K, T, sigma) PLUS the standard
# non-dimensionalised representations (F/K, ln F/K). The raw F and K are
# included so the model can learn strike-specific skew effects that a
# moneyness-only representation hides (see review feedback).
FEATURES_BASE = ['F_dec23', 'strike', 'moneyness', 'log_moneyness', 'T', 'sigma']
FEATURES_IV   = FEATURES_BASE + ['iv_call']
TARGET_CALL   = 'call_delta'
TARGET_PUT    = 'put_delta'

# --- Black-76-inspired engineered features (used by the NN, Sec. 5) ---------
# These encode the analytical structure the network would otherwise have to
# rediscover from raw (F, K, T, sigma). Edge cases (T <= 0, sigma <= 0,
# missing F/K/sigma) are guarded with a small epsilon.
EPS        = 1e-8
T_safe     = np.maximum(df['T'].astype(float).values,     EPS)
sigma_safe = np.maximum(df['sigma'].astype(float).values, EPS)
F_safe     = np.maximum(df['F_dec23'].astype(float).values, EPS)
K_safe     = np.maximum(df['strike'].astype(float).values,  EPS)

df['log_FoverK']        = np.log(F_safe / K_safe)            # = log_moneyness
df['sigma_sqrtT']       = sigma_safe * np.sqrt(T_safe)
df['d1_rv']             = (df['log_FoverK'].values
                            + 0.5 * sigma_safe**2 * T_safe) / df['sigma_sqrtT'].values
df['logFK_over_sigmaT'] = df['log_FoverK'].values / df['sigma_sqrtT'].values

# Replace any non-finite leftovers (defensive)
for _col in ['log_FoverK', 'sigma_sqrtT', 'd1_rv', 'logFK_over_sigmaT']:
    df[_col] = df[_col].replace([np.inf, -np.inf], np.nan).fillna(0.0)

FEATURES_NN = FEATURES_BASE + ['sigma_sqrtT', 'd1_rv', 'logFK_over_sigmaT']

# Back-compat: existing downstream code uses FEATURES/TARGET
FEATURES = FEATURES_BASE
TARGET   = TARGET_CALL

df = df.sort_values(['date', 'strike']).reset_index(drop=True)

unique_dates = df['date'].unique()
split_date   = unique_dates[int(len(unique_dates) * 0.8)]
print(f"Split date: {pd.Timestamp(split_date).date()}")

train_mask = df['date'] < split_date
test_mask  = ~train_mask

# --- Base feature matrix ----------------------------------------------------
X_train = df.loc[train_mask, FEATURES_BASE]
X_test  = df.loc[test_mask,  FEATURES_BASE]
y_train = df.loc[train_mask, TARGET_CALL]
y_test  = df.loc[test_mask,  TARGET_CALL]

# --- NN feature matrix (Black-76-inspired engineered features) -------------
X_train_nn = df.loc[train_mask, FEATURES_NN]
X_test_nn  = df.loc[test_mask,  FEATURES_NN]

# --- Put-delta targets (same X) ---------------------------------------------
y_train_put = df.loc[train_mask, TARGET_PUT]
y_test_put  = df.loc[test_mask,  TARGET_PUT]

# --- IV-augmented feature matrix --------------------------------------------
iv_ok = df['iv_call'].notna()
X_train_iv = df.loc[train_mask & iv_ok, FEATURES_IV]
X_test_iv  = df.loc[test_mask  & iv_ok, FEATURES_IV]
y_train_iv = df.loc[train_mask & iv_ok, TARGET_CALL]
y_test_iv  = df.loc[test_mask  & iv_ok, TARGET_CALL]

# --- Residual targets: market_delta - black_RV_delta ------------------------
# Trains the model on the smile/skew correction layer that an RV-Black hedger
# is missing; reconstructs the effective delta as black_rv + ML_residual.
res_call_train = (df.loc[train_mask, TARGET_CALL]
                  - df.loc[train_mask, 'black_delta']).values
res_call_test  = (df.loc[test_mask,  TARGET_CALL]
                  - df.loc[test_mask,  'black_delta']).values
res_put_train  = (df.loc[train_mask, TARGET_PUT]
                  - df.loc[train_mask, 'black_put_delta']).values
res_put_test   = (df.loc[test_mask,  TARGET_PUT]
                  - df.loc[test_mask,  'black_put_delta']).values

print(f"Base set          -- Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"NN  set (FEATURES_NN={len(FEATURES_NN)} cols) -- Train: {X_train_nn.shape[0]} | Test: {X_test_nn.shape[0]}")
print(f"IV-augmented set  -- Train: {X_train_iv.shape[0]} rows | Test: {X_test_iv.shape[0]} rows")
print(f"IV-subset coverage on test fold: "
      f"{100 * X_test_iv.shape[0] / X_test.shape[0]:5.2f} %")
print(f"Residual target (call): mean={res_call_train.mean():+.4f}  std={res_call_train.std():.4f}")
print(f"Residual target (put) : mean={res_put_train.mean():+.4f}  std={res_put_train.std():.4f}")

# --- Date-blocked K-fold CV iterator ----------------------------------------
# Each fold uses a contiguous chronological block of UNIQUE DATES for the
# validation set. All strikes for each date stay in the same fold, so a date
# is never split across train/val (which a row-count-based TimeSeriesSplit
# would do given 5 strike rows per date).
class DateBlockedKFold:
    def __init__(self, n_splits=5, date_col='date', dates=None):
        self.n_splits = n_splits
        self.date_col = date_col
        self.dates    = dates  # array-like of dates aligned with X.index

    def split(self, X, y=None, groups=None):
        if self.dates is None:
            raise ValueError("DateBlockedKFold requires `dates` aligned with X")
        dates = pd.Series(self.dates).reset_index(drop=True)
        unique = np.array(sorted(dates.unique()))
        # Equal-size contiguous date blocks
        block = np.array_split(unique, self.n_splits)
        for k in range(self.n_splits):
            val_dates = block[k]
            val_mask  = dates.isin(val_dates).values
            tr_mask   = ~val_mask
            yield np.where(tr_mask)[0], np.where(val_mask)[0]

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

train_dates = df.loc[train_mask, 'date'].reset_index(drop=True).values
dbcv        = DateBlockedKFold(n_splits=5, dates=train_dates)
# Same iterator on the IV subset
train_dates_iv = df.loc[train_mask & iv_ok, 'date'].reset_index(drop=True).values
dbcv_iv        = DateBlockedKFold(n_splits=5, dates=train_dates_iv)

# Sanity: print first split sizes
for fi, (tr, va) in enumerate(dbcv.split(X_train)):
    if fi < 2:
        print(f"  fold {fi}: train={len(tr)} val={len(va)}")

# --- Selection-bias check on the IV subset ---------------------------------
test_full = df.loc[test_mask, FEATURES_BASE + ['black_delta']]
test_iv   = df.loc[test_mask & iv_ok, FEATURES_BASE + ['black_delta']]
selbias = pd.concat({
    'full_test':  test_full.describe().loc[['mean', 'std', '50%']],
    'iv_subset':  test_iv.describe().loc[['mean', 'std', '50%']],
}, axis=1).round(4)
print("\nSelection-bias check (full test fold vs IV-extractable subset):")
display(selbias)

In [ ]:
# == 4.2 Ridge & Lasso with date-blocked CV ===================================
results = {}

# --- Ridge Regression ---
ridge_pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge())])
ridge_params = {'ridge__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
ridge_cv = GridSearchCV(ridge_pipe, ridge_params, cv=dbcv,
                        scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_cv.fit(X_train, y_train)
y_pred_ridge = ridge_cv.predict(X_test)
results['Ridge'] = {
    'mae':  mean_absolute_error(y_test, y_pred_ridge),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    'r2':   r2_score(y_test, y_pred_ridge),
    'pred': y_pred_ridge,
}
print(f"Ridge -- best alpha: {ridge_cv.best_params_} | MAE: {results['Ridge']['mae']:.4f}")

# --- Lasso Regression ---
lasso_pipe = Pipeline([('scaler', StandardScaler()), ('lasso', Lasso(max_iter=10000))])
lasso_params = {'lasso__alpha': [1e-4, 1e-3, 0.01, 0.1, 1.0]}
lasso_cv = GridSearchCV(lasso_pipe, lasso_params, cv=dbcv,
                        scoring='neg_mean_absolute_error', n_jobs=-1)
lasso_cv.fit(X_train, y_train)
y_pred_lasso = lasso_cv.predict(X_test)
results['Lasso'] = {
    'mae':  mean_absolute_error(y_test, y_pred_lasso),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_lasso)),
    'r2':   r2_score(y_test, y_pred_lasso),
    'pred': y_pred_lasso,
}
print(f"Lasso -- best alpha: {lasso_cv.best_params_} | MAE: {results['Lasso']['mae']:.4f}")

In [ ]:
# == 4.3 Random Forest & Gradient Boosting ====================================
# --- Random Forest ---
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_params = {
    'n_estimators': [100, 300],
    'max_depth': [5, 10, 20, None],
    'min_samples_leaf': [2, 5],
}
rf_cv = GridSearchCV(rf, rf_params, cv=dbcv,
                     scoring='neg_mean_absolute_error', n_jobs=-1)
rf_cv.fit(X_train, y_train)
y_pred_rf = rf_cv.predict(X_test)
results['Random Forest'] = {
    'mae':  mean_absolute_error(y_test, y_pred_rf),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_rf)),
    'r2':   r2_score(y_test, y_pred_rf),
    'pred': y_pred_rf,
}
print(f"RF -- best: {rf_cv.best_params_} | MAE: {results['Random Forest']['mae']:.4f}")

# --- Gradient Boosting (raw target) ---
gb = GradientBoostingRegressor(random_state=42)
gb_params = {
    'n_estimators': [100, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
}
gb_cv = GridSearchCV(gb, gb_params, cv=dbcv,
                     scoring='neg_mean_absolute_error', n_jobs=-1)
gb_cv.fit(X_train, y_train)
y_pred_gb = gb_cv.predict(X_test)
results['Gradient Boosting'] = {
    'mae':  mean_absolute_error(y_test, y_pred_gb),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_gb)),
    'r2':   r2_score(y_test, y_pred_gb),
    'pred': y_pred_gb,
}
print(f"GB -- best: {gb_cv.best_params_} | MAE: {results['Gradient Boosting']['mae']:.4f}")

# --- Gradient Boosting on the RESIDUAL target -------------------------------
# Train to predict (market_delta - RV-Black delta), then add RV-Black back.
# This frames the ML contribution as the smile/skew correction layer that
# constant-vol Black 76 is structurally missing.
gb_res_cv = GridSearchCV(GradientBoostingRegressor(random_state=42),
                         gb_params, cv=dbcv,
                         scoring='neg_mean_absolute_error', n_jobs=-1)
gb_res_cv.fit(X_train, res_call_train)
res_pred_test = gb_res_cv.predict(X_test)
y_pred_gb_res = df.loc[test_mask, 'black_delta'].values + res_pred_test
results['GB (residual)'] = {
    'mae':  mean_absolute_error(y_test, y_pred_gb_res),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_gb_res)),
    'r2':   r2_score(y_test, y_pred_gb_res),
    'pred': y_pred_gb_res,
}
print(f"GB (residual) -- best: {gb_res_cv.best_params_} | MAE: {results['GB (residual)']['mae']:.4f}")

In [ ]:
# == 4.4 Feature importances (tree-based models) =============================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, model) in zip(axes, [('Random Forest', rf_cv),
                                      ('Gradient Boosting', gb_cv)]):
    imp = model.best_estimator_.feature_importances_
    idx = np.argsort(imp)[::-1]
    ax.barh([FEATURES[i] for i in idx], imp[idx], color='steelblue')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'{name} Feature Importances')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

---
### 4.5 — Extending ML Models to Put Delta

The provided data also contains put delta. Ignoring it loses information and prevents us from empirically testing **put-call parity on deltas** (see Sec. 6.7). We train separate Random Forest and Gradient Boosting regressors on the put target, using the same feature set and the same `TimeSeriesSplit` CV. We prefer *separate* models over a multi-output regressor because:

- Call delta lives on $[0, 1]$, put delta on $[-1, 0]$ — very different marginal distributions.
- Independent hyperparameter tuning tends to give better per-target accuracy when the error surfaces differ (here the two targets are structurally related by parity but empirically distinct in tail behaviour).

A `Black 76 (Put)` benchmark row is also added to `put_results` so that we can compare, strike-by-strike, the analytical put delta against the ML put delta.

In [ ]:
# == 4.5 ML models for put delta ==============================================
put_results = {}

# --- Random Forest on put ---
rf_put = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_put_cv = GridSearchCV(rf_put, rf_params, cv=dbcv,
                          scoring='neg_mean_absolute_error', n_jobs=-1)
rf_put_cv.fit(X_train, y_train_put)
y_pred_rf_put = rf_put_cv.predict(X_test)
put_results['Random Forest'] = {
    'mae':  mean_absolute_error(y_test_put, y_pred_rf_put),
    'rmse': np.sqrt(mean_squared_error(y_test_put, y_pred_rf_put)),
    'r2':   r2_score(y_test_put, y_pred_rf_put),
    'pred': y_pred_rf_put,
}
print(f"RF (put) -- best: {rf_put_cv.best_params_} | MAE: {put_results['Random Forest']['mae']:.4f}")

# --- Gradient Boosting on put ---
gb_put = GradientBoostingRegressor(random_state=42)
gb_put_cv = GridSearchCV(gb_put, gb_params, cv=dbcv,
                          scoring='neg_mean_absolute_error', n_jobs=-1)
gb_put_cv.fit(X_train, y_train_put)
y_pred_gb_put = gb_put_cv.predict(X_test)
put_results['Gradient Boosting'] = {
    'mae':  mean_absolute_error(y_test_put, y_pred_gb_put),
    'rmse': np.sqrt(mean_squared_error(y_test_put, y_pred_gb_put)),
    'r2':   r2_score(y_test_put, y_pred_gb_put),
    'pred': y_pred_gb_put,
}
print(f"GB (put) -- best: {gb_put_cv.best_params_} | MAE: {put_results['Gradient Boosting']['mae']:.4f}")

# --- GB on residual (market_put - RV-Black put) -----------------------------
gb_res_put_cv = GridSearchCV(GradientBoostingRegressor(random_state=42),
                              gb_params, cv=dbcv,
                              scoring='neg_mean_absolute_error', n_jobs=-1)
gb_res_put_cv.fit(X_train, res_put_train)
y_pred_gb_res_put = (df.loc[test_mask, 'black_put_delta'].values
                     + gb_res_put_cv.predict(X_test))
put_results['GB (residual)'] = {
    'mae':  mean_absolute_error(y_test_put, y_pred_gb_res_put),
    'rmse': np.sqrt(mean_squared_error(y_test_put, y_pred_gb_res_put)),
    'r2':   r2_score(y_test_put, y_pred_gb_res_put),
    'pred': y_pred_gb_res_put,
}
print(f"GB (put, residual) -- best: {gb_res_put_cv.best_params_} | MAE: {put_results['GB (residual)']['mae']:.4f}")

# --- RV-Black 76 put delta benchmark ---
black_put_test = df.loc[test_mask, 'black_put_delta'].values
put_results['Black 76 (RV)'] = {
    'mae':  mean_absolute_error(y_test_put, black_put_test),
    'rmse': np.sqrt(mean_squared_error(y_test_put, black_put_test)),
    'r2':   r2_score(y_test_put, black_put_test),
    'pred': black_put_test,
}
print(f"Black 76 (RV, put) -- MAE: {put_results['Black 76 (RV)']['mae']:.4f}")

# --- IV-Black put delta benchmark (analytical with extracted IV) -----------
iv_put_pred = df.loc[test_mask, 'black_iv_put_delta'].values
mask_iv_put = ~np.isnan(iv_put_pred)
put_results['Black 76 (IV)'] = {
    'mae':  mean_absolute_error(y_test_put.values[mask_iv_put], iv_put_pred[mask_iv_put]),
    'rmse': np.sqrt(mean_squared_error(y_test_put.values[mask_iv_put], iv_put_pred[mask_iv_put])),
    'r2':   r2_score(y_test_put.values[mask_iv_put], iv_put_pred[mask_iv_put]),
    'pred': iv_put_pred,  # may contain NaN for failed inversions
}
print(f"Black 76 (IV, put) -- MAE on IV-extractable test fold (n={mask_iv_put.sum()}): "
      f"{put_results['Black 76 (IV)']['mae']:.5f}")

---
### 4.6 — SHAP Feature-Importance Analysis

Gini-based feature importance (Section 4.4) only gives a global ranking. **SHAP** (Lundberg & Lee, 2017) attributes each individual prediction to its input features via Shapley values from cooperative game theory, producing both global and local interpretations. We apply `TreeExplainer` to the best tree model (Gradient Boosting on call delta) and produce:

- A **beeswarm plot** — each dot is a test-set observation, coloured by feature value; horizontal position is the feature's SHAP value (contribution to the prediction). Features are ordered by global importance.
- A **waterfall plot** — the additive decomposition of a single **at-the-money** observation from the expected value to the model's prediction.

This is a more rigorous and local story than the classic Gini ranking and makes it visually obvious why moneyness and time-to-maturity dominate.

In [ ]:
# == 4.6 SHAP feature-importance analysis =====================================
gb_best = gb_cv.best_estimator_
explainer = shap.TreeExplainer(gb_best)

# Subsample for speed
rng = np.random.RandomState(SEED)
sample_idx = rng.choice(len(X_test), size=min(500, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx]
shap_values = explainer.shap_values(X_sample)

# Beeswarm (global)
plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURES_BASE, show=False)
plt.title('SHAP Beeswarm — Gradient Boosting on Call Delta')
plt.tight_layout()
plt.show()

# Waterfall plot for an at-the-money observation
test_df_full = df.loc[test_mask].reset_index(drop=True)
atm_idx = (test_df_full['moneyness'] - 1.0).abs().idxmin()
x_atm = X_test.reset_index(drop=True).iloc[atm_idx:atm_idx+1]

shap_exp = shap.Explanation(
    values=explainer.shap_values(x_atm)[0],
    base_values=explainer.expected_value,
    data=x_atm.iloc[0].values,
    feature_names=FEATURES_BASE,
)
plt.figure(figsize=(10, 5))
shap.plots.waterfall(shap_exp, show=False)
plt.title(f'SHAP Waterfall — ATM observation (F/K={test_df_full.loc[atm_idx, "moneyness"]:.3f}, '
          f'T={test_df_full.loc[atm_idx, "T"]:.3f}y)')
plt.tight_layout()
plt.show()

---
### 4.7 — IV-Augmented Gradient Boosting

We re-train the best tree model (Gradient Boosting) with `iv_call` added to the feature set. If implied volatility is genuinely the "missing information" that explains the gap between the constant-vol Black 76 delta and the market delta, the IV-augmented model should achieve substantially lower MAE than the base variant. We evaluate on the subset of the test window where IV could be extracted.

In [ ]:
# == 4.7 Gradient Boosting with implied volatility as an added feature ========
gb_iv = GradientBoostingRegressor(random_state=42)
gb_iv_cv = GridSearchCV(gb_iv, gb_params, cv=dbcv_iv,
                         scoring='neg_mean_absolute_error', n_jobs=-1)
gb_iv_cv.fit(X_train_iv, y_train_iv)
y_pred_gb_iv = gb_iv_cv.predict(X_test_iv)

results['GB + IV'] = {
    'mae':  mean_absolute_error(y_test_iv, y_pred_gb_iv),
    'rmse': np.sqrt(mean_squared_error(y_test_iv, y_pred_gb_iv)),
    'r2':   r2_score(y_test_iv, y_pred_gb_iv),
    'pred': y_pred_gb_iv,
}
print(f"GB + IV -- best: {gb_iv_cv.best_params_}")
print(f"GB + IV (test n={len(y_test_iv)}) -- MAE: {results['GB + IV']['mae']:.5f}")
print(f"GB base (test n={len(y_test)})    -- MAE: {results['Gradient Boosting']['mae']:.5f}")

# --- Apples-to-apples: also evaluate the base GB on the SAME IV subset -----
gb_base_on_iv_subset = gb_cv.predict(X_test_iv[FEATURES_BASE])
mae_base_on_iv = mean_absolute_error(y_test_iv, gb_base_on_iv_subset)
print(f"\nApples-to-apples on identical IV-extractable test rows:")
print(f"  GB base on IV subset -- MAE: {mae_base_on_iv:.5f}")
print(f"  GB + IV  on IV subset -- MAE: {results['GB + IV']['mae']:.5f}")
print(f"  Net IV-feature gain : {mae_base_on_iv - results['GB + IV']['mae']:+.5f}")
print("\nNote: feeding back-solved IV to a flexible learner does *not* recover the")
print("IV-Black analytic formula in 5x5 hyperparameter search on ~1700 train rows.")
print("The analytical IV-Black benchmark (Sec. 6.1) remains the relevant reference.")

# Feature importances including IV
imp_iv = gb_iv_cv.best_estimator_.feature_importances_
idx = np.argsort(imp_iv)[::-1]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh([FEATURES_IV[i] for i in idx], imp_iv[idx], color='mediumseagreen')
ax.set_xlabel('Feature Importance')
ax.set_title('Gradient Boosting Feature Importances (with IV)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## Section 5 — Deep Learning Model (Keras Feedforward Neural Network)

Using a neural network to approximate the option delta goes beyond the explicit syllabus but is directly requested by Q2. The ability of deep networks to capture non-linear mappings makes them natural candidates for approximating the Black-Scholes delta surface (see Hutchinson, Lo & Poggio, 1994, "A Nonparametric Approach to Pricing and Hedging Derivative Securities Via Learning Networks," *Journal of Finance*, 49(3), 851-889).

### Architecture

| Layer | Neurons | Activation | Notes |
|---|---|---|---|
| Input | 4 | — | moneyness, log-moneyness, tau, sigma |
| Hidden 1 | 64 | ReLU | + BatchNorm + Dropout(0.2) |
| Hidden 2 | 32 | ReLU | + BatchNorm + Dropout(0.2) |
| Hidden 3 | 16 | ReLU | + BatchNorm |
| Output | 1 | Sigmoid | delta in [0, 1] for calls |

- **Loss:** Mean Squared Error
- **Optimizer:** Adam (lr = 1e-3)
- **Early stopping:** patience = 20, monitoring validation loss

In [ ]:
# == 5.1 Prepare data for Keras ================================================
# Use FEATURES_NN (FEATURES_BASE + Black-76-inspired engineered features:
# sigma*sqrt(T), d1_rv, log(F/K)/(sigma*sqrt(T))). The scaler is fit on the
# TRAINING fold only — no test-set leakage.
nn_scaler  = StandardScaler()
X_train_sc = nn_scaler.fit_transform(X_train_nn)
X_test_sc  = nn_scaler.transform(X_test_nn)

# Chronological internal val split (X_train_nn is already sorted by date)
val_split = int(len(X_train_sc) * 0.85)
X_tr, X_val = X_train_sc[:val_split], X_train_sc[val_split:]
y_tr, y_val = y_train.values[:val_split], y_train.values[val_split:]

print(f"DL train: {X_tr.shape[0]} | DL val: {X_val.shape[0]} | Test: {X_test_sc.shape[0]}")
print(f"Input dimension: {X_tr.shape[1]} (FEATURES_NN = {FEATURES_NN})")

In [ ]:
# == 5.2 Build & train the neural network =====================================
# Modest architecture: Dense(64) -> ReLU -> Dropout -> Dense(32) -> ReLU ->
# Dense(1, sigmoid). The sigmoid output keeps the call-delta prediction in
# [0, 1] (futures-style convention), which is a cheap inductive bias. We
# regularise with dropout + L2 and rely on early stopping for capacity
# control. Huber loss is robust to the few outlier quotes near deep ITM.
def build_delta_nn(input_dim, l2=1e-4, dropout=0.20):
    reg = keras.regularizers.l2(l2)
    return keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu', kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu', kernel_regularizer=reg),
        layers.Dense(1, activation='sigmoid'),  # call delta in [0, 1]
    ])

model = build_delta_nn(input_dim=X_tr.shape[1])
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.Huber(delta=0.05),
    metrics=['mae'],
)
model.summary()

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=300,
    batch_size=64,
    callbacks=[
        callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6),
    ],
    verbose=0,
)
print(f"Training stopped at epoch {len(history.history['loss'])}")

In [ ]:
# == 5.3 Training curves =======================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].set_title('Loss Curves')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].set_title('MAE Curves')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# == 5.4 Evaluate on test set ==================================================
y_pred_nn = model.predict(X_test_sc, verbose=0).flatten()

# Validation MAE/MSE (best-epoch values, as restored by EarlyStopping):
best_epoch = int(np.argmin(history.history['val_loss']))
val_mae    = float(history.history['val_mae'][best_epoch])
val_loss   = float(history.history['val_loss'][best_epoch])

results['Neural Network'] = {
    'mae':  mean_absolute_error(y_test, y_pred_nn),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_nn)),
    'r2':   r2_score(y_test, y_pred_nn),
    'pred': y_pred_nn,
    'val_mae':  val_mae,
    'val_loss': val_loss,
}
nn_res = results['Neural Network']
print(f"NN -- best epoch: {best_epoch + 1}/{len(history.history['loss'])}")
print(f"NN -- val   MAE: {val_mae:.4f}  val MSE: {val_loss:.6f}")
print(f"NN -- test  MAE: {nn_res['mae']:.4f}  RMSE: {nn_res['rmse']:.4f}  R2: {nn_res['r2']:.4f}")


---
### 5.5 — Neural Network for Put Delta

For completeness we re-train the same architecture on the put-delta target. Because put delta lives in $[-1, 0]$, we replace the plain `sigmoid` output with a **shifted sigmoid**, $\hat{\Delta}_{\text{put}} = \sigma(z) - 1 \in [-1, 0]$, so the NN output is structurally bounded in the correct range. Otherwise the optimiser, loss and callbacks are identical to Sec. 5.2.

In [ ]:
# == 5.5 Neural network for put delta =========================================
# AI-assistance note: the shifted-sigmoid trick (sigma(z) - 1 to map output to
# [-1, 0]) was suggested by Colab AI; the choice to train a separate put model
# rather than a multi-output one is the authors' (see markdown 4.5).
y_tr_put  = y_train_put.values[:val_split]
y_val_put = y_train_put.values[val_split:]

def build_put_delta_nn(input_dim, l2=1e-4, dropout=0.20):
    '''Same architecture as build_delta_nn but output activation maps to [-1, 0].'''
    reg = keras.regularizers.l2(l2)
    x_in = layers.Input(shape=(input_dim,))
    x = layers.Dense(64, activation='relu', kernel_regularizer=reg)(x_in)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(32, activation='relu', kernel_regularizer=reg)(x)
    raw = layers.Dense(1, activation='sigmoid')(x)                     # in [0, 1]
    out = layers.Lambda(lambda z: z - 1.0, name='put_delta')(raw)      # in [-1, 0]
    return keras.Model(x_in, out)

model_put = build_put_delta_nn(input_dim=X_tr.shape[1])
model_put.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss=keras.losses.Huber(delta=0.05),
    metrics=['mae'],
)
history_put = model_put.fit(
    X_tr, y_tr_put,
    validation_data=(X_val, y_val_put),
    epochs=300, batch_size=64,
    callbacks=[
        callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6),
    ],
    verbose=0,
)
y_pred_nn_put = model_put.predict(X_test_sc, verbose=0).flatten()

best_epoch_put = int(np.argmin(history_put.history['val_loss']))
val_mae_put    = float(history_put.history['val_mae'][best_epoch_put])

put_results['Neural Network'] = {
    'mae':  mean_absolute_error(y_test_put, y_pred_nn_put),
    'rmse': np.sqrt(mean_squared_error(y_test_put, y_pred_nn_put)),
    'r2':   r2_score(y_test_put, y_pred_nn_put),
    'pred': y_pred_nn_put,
    'val_mae': val_mae_put,
}
print(f"NN (put) -- best epoch: {best_epoch_put + 1}/{len(history_put.history['loss'])}")
print(f"NN (put) -- val   MAE: {val_mae_put:.4f}")
print(f"NN (put) -- test  MAE: {put_results['Neural Network']['mae']:.4f}  "
      f"RMSE: {put_results['Neural Network']['rmse']:.4f}  "
      f"R2: {put_results['Neural Network']['r2']:.4f}")

---
### 5.6 — Residual Neural Network

The raw network in Sec. 5.2/5.5 has to relearn the analytical structure of $\Delta = N(d_1)$ from a few thousand panel rows. A more economically grounded use of deep learning is to **start from the analytical Black-76 prediction and learn only the smile/skew correction**. We train the network on the residual

$$ r = \Delta^{\text{market}} - \Delta^{\text{RV-Black}} $$

and reconstruct the model delta as $\Delta^{NN-residual} = \Delta^{\text{RV-Black}} + \hat r_{NN}$. This mirrors the residual gradient-boosting set-up in Sec. 4.3, lets the NN focus its capacity on what Black-76 cannot capture (smile, skew, rate/quote frictions), and uses **the same chronological split, scaler (fit on TRAIN only), and Black-76-inspired features** as the raw NN.

In [ ]:
# == 5.6 Residual neural network -- calls and puts ============================
# Trains a small NN on r = market_delta - RV-Black delta, then adds RV-Black
# back. Same scaler / same chronological internal val split as Sec. 5.1 --
# nothing about the train/test partition changes.
def build_residual_nn(input_dim, l2=1e-4, dropout=0.20):
    reg = keras.regularizers.l2(l2)
    return keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu', kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu', kernel_regularizer=reg),
        layers.Dense(1, activation='linear'),  # residuals are small and signed
    ])

def fit_residual_nn(X_tr_, y_tr_, X_val_, y_val_, *, loss=None):
    if loss is None:
        loss = keras.losses.Huber(delta=0.02)
    m = build_residual_nn(input_dim=X_tr_.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=loss, metrics=['mae'])
    h = m.fit(X_tr_, y_tr_,
              validation_data=(X_val_, y_val_),
              epochs=300, batch_size=64,
              callbacks=[
                  callbacks.EarlyStopping(patience=20, restore_best_weights=True),
                  callbacks.ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6),
              ],
              verbose=0)
    return m, h

# --- CALL: residual target --------------------------------------------------
res_call_tr  = res_call_train[:val_split]
res_call_val = res_call_train[val_split:]
m_res_call, h_res_call = fit_residual_nn(X_tr, res_call_tr, X_val, res_call_val)
r_hat_call = m_res_call.predict(X_test_sc, verbose=0).flatten()
y_pred_nn_res_call = df.loc[test_mask, 'black_delta'].values + r_hat_call

results['NN (residual)'] = {
    'mae':  mean_absolute_error(y_test, y_pred_nn_res_call),
    'rmse': np.sqrt(mean_squared_error(y_test, y_pred_nn_res_call)),
    'r2':   r2_score(y_test, y_pred_nn_res_call),
    'pred': y_pred_nn_res_call,
}
print(f"NN (residual, call) -- best epoch: "
      f"{int(np.argmin(h_res_call.history['val_loss'])) + 1}/"
      f"{len(h_res_call.history['loss'])}")
print(f"NN (residual, call) -- test MAE: {results['NN (residual)']['mae']:.5f}  "
      f"RMSE: {results['NN (residual)']['rmse']:.5f}  R2: {results['NN (residual)']['r2']:.4f}")

# --- PUT: residual target ---------------------------------------------------
res_put_tr  = res_put_train[:val_split]
res_put_val = res_put_train[val_split:]
m_res_put, h_res_put = fit_residual_nn(X_tr, res_put_tr, X_val, res_put_val)
r_hat_put = m_res_put.predict(X_test_sc, verbose=0).flatten()
y_pred_nn_res_put = df.loc[test_mask, 'black_put_delta'].values + r_hat_put

put_results['NN (residual)'] = {
    'mae':  mean_absolute_error(y_test_put, y_pred_nn_res_put),
    'rmse': np.sqrt(mean_squared_error(y_test_put, y_pred_nn_res_put)),
    'r2':   r2_score(y_test_put, y_pred_nn_res_put),
    'pred': y_pred_nn_res_put,
}
print(f"NN (residual, put)  -- test MAE: {put_results['NN (residual)']['mae']:.5f}  "
      f"RMSE: {put_results['NN (residual)']['rmse']:.5f}  R2: {put_results['NN (residual)']['r2']:.4f}")

In [ ]:
# == 5.6b Compact comparison: residual NN vs benchmarks =======================
def _row(dct, key):
    if key not in dct:
        return {'MAE': float('nan'), 'RMSE': float('nan')}
    return {'MAE': dct[key]['mae'], 'RMSE': dct[key]['rmse']}

_rows = ['Black 76 (RV)', 'Neural Network', 'GB (residual)',
         'NN (residual)', 'Black 76 (IV)']
_compact = pd.DataFrame({
    name: {
        'Call MAE':  _row(results,     name)['MAE'],
        'Call RMSE': _row(results,     name)['RMSE'],
        'Put MAE':   _row(put_results, name)['MAE'],
        'Put RMSE':  _row(put_results, name)['RMSE'],
    } for name in _rows
}).T

print("Compact comparison: residual NN vs raw NN, residual GB, RV-Black, IV-Black")
display(_compact.round(5))

---
### 5.7 — Discussion: how to interpret the NN results

**(1) Why a raw NN may underperform Black-76.** Black-76 already encodes the no-arbitrage structure ($\Delta = N(d_1)$ in our chosen futures-style convention), with the right tail behaviour at deep ITM/OTM strikes built in. A small, generic feed-forward network has to relearn that geometry from a few thousand panel rows, and it gives away calibration accuracy in regions of the strike grid that the formula already gets right. With ~5 strikes per date and a relatively short history, the raw NN is in a regime where a strong analytical prior dominates.

**(2) Why residual learning is more economically sensible.** The market delta differs from RV-Black mainly because of (i) the volatility skew (constant historical vol vs strike-dependent implied vol), (ii) the discount-factor / convention quirk, and (iii) microstructure noise. Training the NN on $r = \Delta^{\text{market}} - \Delta^{\text{RV-Black}}$ frames the deep-learning problem as the *correction layer* the analytical model is structurally missing — preserving the economically motivated baseline and using NN flexibility only where it adds value. Combined with the engineered features $\sigma\sqrt T,\, d_1^{RV},\, \log(F/K)/(\sigma\sqrt T)$, the network sees the same coordinates as the analytical formula.

**(3) Does the residual NN improve over RV-Black?** The compact table in 5.6 reports the residual NN's call/put MAE versus RV-Black on the same chronological test fold. Any positive improvement here is the smile/skew correction the network has been able to learn — i.e. the part of the market delta that constant-vol Black 76 cannot reach.

**(4) Does it beat residual GB?** Residual gradient boosting is a strong tabular baseline on a small panel ($\sim$5 strikes per day). A regularised residual NN can match it, but on this dataset we should not expect it to dominate: trees are well-suited to the piecewise-monotonic skew correction, and the NN's edge typically requires denser strike grids and longer histories. The take-away is that *residualisation*, not the choice of ML estimator, is what closes most of the gap with the analytical benchmark.

**(5) Why IV-Black remains a natural upper benchmark.** IV-Black uses **option-specific implied volatility**, back-solved from the market price. It is not a forecasting model — it consumes information from the option chain that the other models do not see — and so the IV-Black MAE is essentially the irreducible bid–ask / quote-rounding floor. It bounds how good *any* model can be while still being supervised on the vendor-published delta.

> The neural network should not be interpreted as a replacement for Black-76. A more sensible use of deep learning is to learn the residual correction to RV-Black. This combines the no-arbitrage structure of the analytical model with the flexibility of a neural network.

---
### 5.8 — Robustness check: walk-forward validation

The chronological train/test split fixed throughout this notebook gives one realisation of the residual NN's out-of-sample performance. To check that the gain (or loss) we read off the test fold is not a chance feature of that single split, we run a small **walk-forward** evaluation: training stops at date $t$, the model is fit on $[d_0, t)$ and scored on the next chronological block $[t, t + \Delta]$, and the train window is then expanded to include that block before the next step. We do **not** replace the main test split — this is a robustness check only, focused on calls.

In [ ]:
# == 5.8 Walk-forward robustness check (residual NN, calls) ===================
# Re-uses the FULL panel and rolls forward in time. Each step:
#   - scaler fit on the TRAIN window only
#   - residual NN fit on r = market_delta - RV-Black delta
#   - scored on the next chronological block (RV-Black + r_hat)
unique_dates_all = np.array(sorted(df['date'].unique()))
n_steps = 4
date_blocks = np.array_split(unique_dates_all, n_steps + 1)

wf_rows = []
running_train_dates = list(date_blocks[0])
for k in range(1, n_steps + 1):
    val_dates = date_blocks[k]
    tr_dates  = np.array(running_train_dates)
    tr_mask = df['date'].isin(tr_dates)
    va_mask = df['date'].isin(val_dates)
    if tr_mask.sum() < 50 or va_mask.sum() < 10:
        running_train_dates.extend(list(val_dates))
        continue
    sc = StandardScaler()
    Xtr = sc.fit_transform(df.loc[tr_mask, FEATURES_NN])
    Xva = sc.transform(    df.loc[va_mask, FEATURES_NN])
    rtr = (df.loc[tr_mask, TARGET_CALL] - df.loc[tr_mask, 'black_delta']).values
    yva = df.loc[va_mask, TARGET_CALL].values
    bk  = df.loc[va_mask, 'black_delta'].values

    m = build_residual_nn(input_dim=Xtr.shape[1])
    m.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss=keras.losses.Huber(delta=0.02), metrics=['mae'])
    m.fit(Xtr, rtr,
          epochs=200, batch_size=64,
          validation_split=0.15,
          callbacks=[callbacks.EarlyStopping(patience=15, restore_best_weights=True)],
          verbose=0)
    pred = bk + m.predict(Xva, verbose=0).flatten()

    mae_nn  = mean_absolute_error(yva, pred)
    mae_blk = mean_absolute_error(yva, bk)
    wf_rows.append({
        'fold':         k,
        'train_end':    pd.Timestamp(tr_dates[-1]).date(),
        'test_start':   pd.Timestamp(val_dates[0]).date(),
        'test_end':     pd.Timestamp(val_dates[-1]).date(),
        'n_train':      int(tr_mask.sum()),
        'n_test':       int(va_mask.sum()),
        'MAE_RV-Black':    mae_blk,
        'MAE_NN-residual': mae_nn,
        'rel_improv_%':    (1 - mae_nn / mae_blk) * 100 if mae_blk else float('nan'),
    })
    running_train_dates.extend(list(val_dates))  # expanding window

wf = pd.DataFrame(wf_rows).set_index('fold')
print("Walk-forward residual NN vs RV-Black (calls):")
display(wf.round(5))
if len(wf):
    n_better = int((wf['rel_improv_%'] > 0).sum())
    print(f"Mean MAE improvement over folds: {wf['rel_improv_%'].mean():+.2f} %  "
          f"(folds where NN beats RV-Black: {n_better}/{len(wf)})")

---
## Section 6 — Comparison: ML/DL Predicted Delta vs Black 76 Delta

We now compare all model predictions against both the **market delta** (target) and the **Black 76 delta** (analytical benchmark) on the held-out test set.

In [ ]:
# == 6.1 Summary tables ========================================================
black_test = df.loc[test_mask, 'black_delta'].values
results['Black 76 (RV)'] = {
    'mae':  mean_absolute_error(y_test, black_test),
    'rmse': np.sqrt(mean_squared_error(y_test, black_test)),
    'r2':   r2_score(y_test, black_test),
    'pred': black_test,
}

# IV-Black analytical benchmark on the test fold (only IV-extractable rows)
iv_test_pred = df.loc[test_mask, 'black_iv_delta'].values
mask_iv      = ~np.isnan(iv_test_pred)
results['Black 76 (IV)'] = {
    'mae':  mean_absolute_error(y_test.values[mask_iv], iv_test_pred[mask_iv]),
    'rmse': np.sqrt(mean_squared_error(y_test.values[mask_iv], iv_test_pred[mask_iv])),
    'r2':   r2_score(y_test.values[mask_iv], iv_test_pred[mask_iv]),
    'pred': iv_test_pred,  # NaNs preserved for length-aligned plots
}

summary = pd.DataFrame({
    name: {'MAE': v['mae'], 'RMSE': v['rmse'], 'R2': v['r2']}
    for name, v in results.items()
}).T.sort_values('MAE')

summary_put = pd.DataFrame({
    name: {'MAE': v['mae'], 'RMSE': v['rmse'], 'R2': v['r2']}
    for name, v in put_results.items()
}).T.sort_values('MAE')

print("CALL DELTA -- Model Comparison on Test Set (Market Delta as Target):")
display(summary.round(5))

print("\nPUT DELTA -- Model Comparison on Test Set (Market Delta as Target):")
display(summary_put.round(5))

# Relative-improvement vs RV-Black benchmark for headline use.
def _rel_table(tbl, ref):
    if ref not in tbl.index:
        return pd.DataFrame()
    base = tbl.loc[ref, 'MAE']
    rel = (1 - tbl['MAE'] / base) * 100
    return rel.rename(f'MAE improvement vs {ref} (%)').to_frame()

print("\nRelative improvement vs RV-Black 76 (calls):")
display(_rel_table(summary, 'Black 76 (RV)').round(2))
print("Relative improvement vs RV-Black 76 (puts):")
display(_rel_table(summary_put, 'Black 76 (RV)').round(2))

In [ ]:
# == 6.2 Scatter plots: predicted vs market delta =============================
model_names = list(results.keys())
n = len(model_names)
ncols = 4
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.atleast_1d(axes).flatten()

for i, name in enumerate(model_names):
    ax = axes[i]
    pred = np.asarray(results[name]['pred'])
    yv   = y_test.values
    if len(pred) != len(yv):
        # IV-restricted entries (fewer rows than full test fold)
        ok = np.ones(len(pred), dtype=bool)
        yv = yv[ok]
    mask = np.isfinite(pred) & np.isfinite(yv) if len(pred)==len(yv) else None
    if mask is not None:
        ax.scatter(pred[mask], yv[mask], s=5, alpha=0.4, c='steelblue')
    else:
        ax.scatter(pred, yv, s=5, alpha=0.4, c='steelblue')
    ax.plot([0, 1], [0, 1], 'r--', lw=1.2)
    ax.set_xlabel('Predicted Delta')
    ax.set_ylabel('Market Delta')
    mae_val = results[name]['mae']
    ax.set_title(f'{name}  (MAE={mae_val:.4f})')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Predicted Delta vs Market Delta (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# == 6.3 Residual analysis by moneyness =======================================
test_df = df.loc[test_mask].reset_index(drop=True).copy()

ncols = 4
nrows = int(np.ceil(len(model_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.atleast_1d(axes).flatten()

for i, name in enumerate(model_names):
    ax = axes[i]
    pred = np.asarray(results[name]['pred'])
    yv   = y_test.values
    if len(pred) != len(yv):
        # Drop NaN residuals (IV inversion failures aligned with full test)
        residuals = yv - pred
    else:
        residuals = yv - pred
    mask = np.isfinite(residuals)
    ax.scatter(test_df['moneyness'].values[mask], residuals[mask],
               s=5, alpha=0.4, c='darkorange')
    ax.axhline(0, color='black', ls='--', lw=1)
    ax.set_xlabel('Moneyness (F/K)')
    ax.set_ylabel('Residual (Market - Predicted)')
    ax.set_title(f'{name}')

for j in range(len(model_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Prediction Residuals vs Moneyness (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# == 6.4 Residual analysis by time to maturity ================================
ncols = 4
nrows = int(np.ceil(len(model_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.atleast_1d(axes).flatten()

for i, name in enumerate(model_names):
    ax = axes[i]
    pred = np.asarray(results[name]['pred'])
    yv   = y_test.values
    residuals = yv - pred
    mask = np.isfinite(residuals)
    ax.scatter(test_df['T'].values[mask], residuals[mask],
               s=5, alpha=0.4, c='seagreen')
    ax.axhline(0, color='black', ls='--', lw=1)
    ax.set_xlabel('Time to Maturity (years)')
    ax.set_ylabel('Residual (Market - Predicted)')
    ax.set_title(f'{name}')

for j in range(len(model_names), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Prediction Residuals vs Time to Maturity (Test Set)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# == 6.5 ML/DL delta vs RV-Black delta (direct comparison) ====================
ml_models = [m for m in model_names if not m.startswith('Black 76')]
ncols = 4
nrows = int(np.ceil(len(ml_models) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = np.atleast_1d(axes).flatten()

for i, name in enumerate(ml_models):
    ax = axes[i]
    pred = np.asarray(results[name]['pred'])
    if len(pred) == len(black_test):
        ax.scatter(black_test, pred, s=5, alpha=0.4, c='purple')
    ax.plot([0, 1], [0, 1], 'r--', lw=1.2)
    ax.set_xlabel('RV-Black 76 Delta')
    ax.set_ylabel(f'{name} Delta')
    ax.set_title(f'{name} vs RV-Black 76')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    ax.set_aspect('equal')

for j in range(len(ml_models), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('ML/DL Delta vs RV-Black 76 Analytical Delta', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
### 6.6 — Strike-by-Strike MAE

Overall MAE can hide very different accuracy profiles across strikes. A model that is good on-the-money but poor deep-OTM can still win in aggregate if the panel is ATM-heavy. Since the five strikes (4000/4200/4400/4600/4800) cover a wide moneyness range, we break down MAE per strike for both calls and puts. This directly addresses the question's "*where does the Black delta fail most?*" angle and is the cleanest diagnostic for the volatility-smile story (Sec. 7.2).

In [ ]:
# == 6.6 Strike-by-strike MAE =================================================
test_df_full = df.loc[test_mask].reset_index(drop=True).copy()
strikes_all  = sorted(test_df_full['strike'].unique())

def _mae_safe(y, p):
    mask = np.isfinite(y) & np.isfinite(p)
    return mean_absolute_error(y[mask], p[mask]) if mask.any() else np.nan

rows = []
for name, v in results.items():
    p = np.asarray(v['pred'])
    if len(p) != len(test_df_full):
        # IV-subsample rows: skip the strike table (would need a separate index)
        continue
    for K in strikes_all:
        m = (test_df_full['strike'] == K).values
        rows.append({'model': name, 'strike': K,
                     'MAE': _mae_safe(y_test.values[m], p[m])})
mae_call_by_K = (pd.DataFrame(rows)
                  .pivot(index='strike', columns='model', values='MAE')
                  .round(5))
print("CALL delta MAE by strike:")
display(mae_call_by_K)

rows_p = []
for name, v in put_results.items():
    p = np.asarray(v['pred'])
    if len(p) != len(test_df_full):
        continue
    for K in strikes_all:
        m = (test_df_full['strike'] == K).values
        rows_p.append({'model': name, 'strike': K,
                       'MAE': _mae_safe(y_test_put.values[m], p[m])})
mae_put_by_K = (pd.DataFrame(rows_p)
                 .pivot(index='strike', columns='model', values='MAE')
                 .round(5))
print("\nPUT delta MAE by strike:")
display(mae_put_by_K)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
mae_call_by_K.plot(kind='bar', ax=axes[0], width=0.8, colormap='tab10')
axes[0].set_title('Call Delta MAE by Strike')
axes[0].set_ylabel('MAE')
axes[0].set_xlabel('Strike K')
axes[0].legend(fontsize=8)
mae_put_by_K.plot(kind='bar', ax=axes[1], width=0.8, colormap='tab10')
axes[1].set_title('Put Delta MAE by Strike')
axes[1].set_ylabel('MAE')
axes[1].set_xlabel('Strike K')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

---
### 6.7 — Put-Call Parity on Deltas

Differentiating the Black 76 put-call parity relation $C - P = e^{-rT}(F - K)$ with respect to $F$ gives the **parity condition on deltas**:

$$\Delta_{\mathrm{call}} - \Delta_{\mathrm{put}} = \begin{cases} e^{-rT} & \text{premium-paid convention} \\ 1 & \text{futures-style convention} \end{cases}$$

This gives us a second (independent of Sec. 3.1) test to verify *which convention* the CME market data uses. Any data-generating process that obeys no-arbitrage must satisfy one of these two identities exactly; the deviation is an empirical diagnostic.

We also plot the **ML-implied parity** $(\hat{\Delta}^{ML}_{\mathrm{call}} - \hat{\Delta}^{ML}_{\mathrm{put}})$. Because our ML models were trained independently on each target, there is no hard-coded constraint guaranteeing parity — a model that has truly learned the delta function should still reproduce it approximately.

In [ ]:
# == 6.7 Put-call parity on deltas — confirmation of Sec. 3.1 choice ==========
parity_market = df['call_delta'] - df['put_delta']
theor_pp      = np.exp(-R_F * df['T'])
theor_fs      = np.ones_like(df['T'])

mae_vs_pp = (parity_market - theor_pp).abs().mean()
mae_vs_fs = (parity_market - theor_fs).abs().mean()
print(f"Mean |Δc − Δp − e^(-rT)|  (premium-paid theory)  : {mae_vs_pp:.5f}")
print(f"Mean |Δc − Δp − 1|        (futures-style theory) : {mae_vs_fs:.5f}")
winner = 'futures-style (= 1)' if mae_vs_fs < mae_vs_pp else 'premium-paid (= e^(-rT))'
print(f"==> Confirms the convention adopted in Sec. 3.1: {CHOSEN_CONVENTION}")
print("    (this is the same vol-free diagnostic that drove the choice upstream)")

# --- Plot empirical parity curve vs both theoretical conventions ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Full dataset (Empirical Parity)
ax = axes[0]
ax.scatter(df['T'], parity_market, s=7, alpha=0.35, label=r'Market: $\Delta_c - \Delta_p$')
T_full_grid = np.linspace(df['T'].min(), df['T'].max(), 100)
ax.plot(T_full_grid, np.exp(-R_F * T_full_grid), color='red',   lw=2, label=r'$e^{-rT}$ (premium-paid)')
ax.axhline(1.0,                color='green', lw=2, label='$1.0$ (futures-style)')
ax.set_xlabel(r'$\tau$ (years)')
ax.set_ylabel(r'$\Delta_{\mathrm{call}} - \Delta_{\mathrm{put}}$')
ax.set_title('Empirical Put-Call Parity on Deltas (Full Sample)')
ax.legend()

# Plot 2: Test set only (ML-Implied Parity)
# Note: This looks shorter because the test set is the last 20% of time-sorted data.
ax = axes[1]
parity_gb_test = results['Gradient Boosting']['pred'] - put_results['Gradient Boosting']['pred']
parity_nn_test = results['Neural Network']['pred']    - put_results['Neural Network']['pred']
parity_mkt_tst = y_test.values - y_test_put.values
T_test = df.loc[test_mask, 'T'].values

ax.scatter(T_test, parity_mkt_tst,  s=6, alpha=0.35, c='steelblue', label='Market (Test)')
ax.scatter(T_test, parity_gb_test,  s=6, alpha=0.35, c='tab:orange', label='GB call $-$ GB put')
ax.scatter(T_test, parity_nn_test,  s=6, alpha=0.35, c='tab:green',  label='NN call $-$ NN put')

# Theoretical lines scaled to test set range for clarity
T_test_grid = np.linspace(T_test.min(), T_test.max(), 100)
ax.plot(T_test_grid, np.exp(-R_F * T_test_grid), color='red',   lw=1.5, ls='--', label=r'$e^{-rT}$')
ax.axhline(1.0, color='green', lw=1.5, ls='--', label='1.0')

ax.set_xlabel(r'$\tau$ (years)')
ax.set_ylabel(r'$\Delta_{\mathrm{call}} - \Delta_{\mathrm{put}}$')
ax.set_title('ML-Implied Parity vs Market (Test Set Only)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\nGB parity RMSE vs market parity (Test Set): "
      f"{np.sqrt(np.mean((parity_gb_test - parity_mkt_tst)**2)):.5f}")
print(f"NN parity RMSE vs market parity (Test Set): "
      f"{np.sqrt(np.mean((parity_nn_test - parity_mkt_tst)**2)):.5f}")

---
### 6.8 — 3D Delta Surface Comparison

The most visually compelling comparison between the ML-learned delta function and the Black 76 analytical delta is to view them both as **surfaces** over the two main drivers — moneyness $F/K$ and time-to-maturity $\tau$ — with volatility fixed at its median value.

- **Left panel:** Gradient Boosting predicted delta $\hat{\Delta}^{GB}(F/K,\tau \mid \bar\sigma)$.
- **Right panel:** Black 76 analytical delta using the convention adopted in Sec. 3.1.

We also plot the **contour of the difference** — positive values mean GB assigns a larger delta than Black 76 at that point of the grid. Such a surface is informative because:
1. The structural "S-shape" in moneyness should appear in both.
2. Near-expiry ($\tau \to 0$) behaviour should steepen sharply in both.
3. Deviations from Black 76 reveal what the ML model has learned **beyond** the constant-vol analytical benchmark — typically the skew/smile curvature captured implicitly.

In [ ]:
# == 6.8 3D delta surface: GB vs RV-Black 76 ==================================
sigma_med = float(df['sigma'].median())
F_med     = float(df['F_dec23'].median())
print(f"Surface plotted at fixed sigma = median = {sigma_med:.4f} and F = {F_med:.2f}")

m_grid = np.linspace(0.85, 1.15, 50)
T_grid = np.linspace(max(5/252, df['T'].min()), df['T'].max(), 50)
M, Tg = np.meshgrid(m_grid, T_grid)
logM = np.log(M)
sig_g = np.full_like(M, sigma_med)
# Strike implied by F_med and the M grid
K_g = F_med / M
F_g = np.full_like(M, F_med)

grid_df = pd.DataFrame({
    'F_dec23':       F_g.ravel(),
    'strike':        K_g.ravel(),
    'moneyness':     M.ravel(),
    'log_moneyness': logM.ravel(),
    'T':             Tg.ravel(),
    'sigma':         sig_g.ravel(),
})

gb_surf = gb_cv.predict(grid_df[FEATURES_BASE]).reshape(M.shape)
d1_grid = (logM + 0.5 * sig_g**2 * Tg) / (sig_g * np.sqrt(Tg))
if CHOSEN_CONVENTION.startswith('futures'):
    bk_surf = norm.cdf(d1_grid)
else:
    bk_surf = np.exp(-R_F * Tg) * norm.cdf(d1_grid)

# --- 3D paired surface plot ---
fig = plt.figure(figsize=(15, 6))
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(M, Tg, gb_surf, cmap='viridis', edgecolor='none', alpha=0.95)
ax1.set_xlabel('Moneyness $F/K$')
ax1.set_ylabel(r'$\tau$ (years)')
ax1.set_zlabel(r'Call $\Delta$')
ax1.set_title('Gradient Boosting Delta Surface')

ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.plot_surface(M, Tg, bk_surf, cmap='viridis', edgecolor='none', alpha=0.95)
ax2.set_xlabel('Moneyness $F/K$')
ax2.set_ylabel(r'$\tau$ (years)')
ax2.set_zlabel(r'Call $\Delta$')
ax2.set_title(f'RV-Black 76 Delta Surface\n({CHOSEN_CONVENTION})')
plt.tight_layout()
plt.show()

# --- Contour plot of the difference ---
fig, ax = plt.subplots(figsize=(10, 5))
diff = gb_surf - bk_surf
vmax = np.abs(diff).max()
cs = ax.contourf(M, Tg, diff, 20, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(cs, ax=ax, label='GB $-$ RV-Black 76')
ax.set_xlabel('Moneyness $F/K$')
ax.set_ylabel(r'$\tau$ (years)')
ax.set_title('Empirical minus Analytical Delta Surface (difference)')
plt.tight_layout()
plt.show()

print(f"Max |GB - Black| over grid: {np.abs(diff).max():.4f}")
print(f"Mean |GB - Black| over grid: {np.abs(diff).mean():.4f}")

---
### 6.9 — Self-Financing Replication: Empirical Test

This is the **core financial test** of the question. A Black 76-consistent call option payoff should be replicable with a self-financing portfolio that is long $\Delta$ units of the underlying futures. Equivalently, "long one call + short $\Delta$ futures" should behave like a cash position earning the risk-free rate on the call's value.

For each business day $t$ (and each strike) we compute the daily hedge error
$$\varepsilon_t(\Delta) = \underbrace{\big(C_{t+1} - C_t\big)}_{\text{call P\&L}} - \underbrace{\Delta_t\,\big(F_{t+1} - F_t\big)}_{\text{hedge P\&L}} - \underbrace{r\,C_t\,\tfrac{1}{252}}_{\text{risk-free earn.}}$$

A perfect delta would make $\varepsilon_t(\Delta) \equiv 0$. We compute $\varepsilon_t$ using three different deltas:

- `delta_black` — Black 76 analytical (chosen convention from Sec. 3.1).
- `delta_gb` — Gradient Boosting prediction.
- `delta_nn` — Neural Network prediction.

We then plot the **cumulative** hedge error (what a trader would see as portfolio drift) per strike, and tabulate the **daily tracking RMSE** and end-of-window accumulated error.

**How to read the plots:** the curve closer to the dashed zero line ("ideal self-financing") corresponds to a delta that replicates the payoff more faithfully. Lower daily tracking RMSE = smaller unhedged residual risk, i.e. a better approximation of the true risk-neutral hedge ratio.

In [ ]:
# == 6.9 Self-financing replication — empirical P&L test ======================
# AI-assistance note: scaffolding for the cumulative-error plot was AI-drafted;
# the choice of comparators (market quoted, IV-Black, RV-Black, GB, NN) and
# the softened economic interpretation are the authors' own.
#
# Hedge identity (Black 76, perfect delta):
#   err_t = (C_{t+1}-C_t) - delta_t*(F_{t+1}-F_t) - r*C_t*dt  ->  0
# Deviations conflate: (i) discrete-rebalancing/gamma error, (ii) vega error
# from sigma drift, and (iii) genuine delta-model error. "Lower RMSE = better"
# is therefore *suggestive*, not definitive — the IV-Black ceiling is not the
# zero-error ceiling.
repl_df = df.loc[test_mask].reset_index(drop=True).copy()
repl_df['delta_market']   = y_test.values
repl_df['delta_gb']       = results['Gradient Boosting']['pred']
repl_df['delta_gb_res']   = results['GB (residual)']['pred']
repl_df['delta_nn']       = results['Neural Network']['pred']
repl_df['delta_rv_black'] = df.loc[test_mask, 'black_delta'].values
repl_df['delta_iv_black'] = df.loc[test_mask, 'black_iv_delta'].values

def hedge_error_series(sub, delta_col, r=R_F, dt=1/252):
    s = sub.sort_values('date').reset_index(drop=True)
    dC = s['call_price'].diff().shift(-1)
    dF = s['F_dec23'].diff().shift(-1)
    err = dC - s[delta_col] * dF - r * s['call_price'] * dt
    return err, s['date']

strike_list = sorted(repl_df['strike'].unique())
delta_cols  = [
    ('delta_market',   'market',    'tab:gray'),
    ('delta_iv_black', 'iv_black',  'tab:purple'),
    ('delta_rv_black', 'rv_black',  'tab:red'),
    ('delta_gb',       'gb',        'tab:blue'),
    ('delta_gb_res',   'gb_res',    'tab:cyan'),
    ('delta_nn',       'nn',        'tab:green'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=False)
axes = axes.flatten()
track_rows = []

for i, K in enumerate(strike_list):
    ax = axes[i]
    sub = repl_df[repl_df['strike'] == K]
    for dcol, label, color in delta_cols:
        err, dates = hedge_error_series(sub, dcol)
        cum = err.fillna(0).cumsum()
        ax.plot(dates, cum, color=color, lw=1.2, label=label)
        track_rows.append({
            'strike': K, 'delta': label,
            'tracking_RMSE':  float(np.sqrt(np.nanmean(err.values**2))),
            'final_cum_err':  float(cum.iloc[-1]),
        })
    ax.axhline(0, color='black', lw=0.8, ls=':')
    ax.set_title(f'Cumulative Hedge Error — K={K}')
    ax.set_ylabel('Cumulative $ / index point')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=7)

for j in range(len(strike_list), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Self-Financing Replication: Cumulative Hedge Error by Strike',
             fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

te_df  = pd.DataFrame(track_rows)
te_rmse = te_df.pivot(index='strike', columns='delta', values='tracking_RMSE').round(3)
te_cum  = te_df.pivot(index='strike', columns='delta', values='final_cum_err').round(2)
print("Daily Hedge-Error RMSE by Strike (index points / day):")
display(te_rmse)
print("\nCumulative Hedge Error at End of Test Window:")
display(te_cum)

print("\nInterpretation (with caveats):")
print("  A perfect, continuously-rebalanced delta would zero out the daily P&L")
print("  of (long call - delta*futures), modulo r*C*dt. In the discrete world")
print("  the residual also captures gamma error (sigma*sqrt(dt) curvature) and")
print("  vega error (drift in sigma). A *lower* tracking RMSE is therefore")
print("  consistent with — but not proof of — a delta closer to the true")
print("  risk-neutral hedge ratio. Market-quoted and IV-Black deltas serve as")
print("  the practical floor; ML/DL deltas should be judged against those.")

---
## Section 7 — Economic Interpretation & Conclusions

> **Numerical anchors.** All MAE / hedge-RMSE figures cited below are printed
> by the *headline numerical results* code cell (Sec. 7.7) so the narrative
> always reflects the latest run; we cite **representative magnitudes** in
> prose and refer to the printed tables for exact decimals.

### 7.1 — Self-Financing Replicating Portfolio

A fundamental result in option-pricing theory (Black & Scholes, 1973; Black, 1976) is that an option payoff can be replicated by a **self-financing portfolio** invested in the underlying security and in the risk-free asset. For a **call on a futures contract** under Black 76:
- Buy the call at price $C$.
- Short $\Delta$ futures.
- The combined portfolio should earn the risk-free rate on an amount equal to $C$.

Section 6.9 tests this empirically by computing the daily hedge error
$\varepsilon_t = (C_{t+1}-C_t) - \Delta_t(F_{t+1}-F_t) - rC_t/252$
for six $\Delta$-estimators: the quoted vendor delta (control), the IV-Black analytic delta, the RV-Black analytic delta, the raw-target Gradient Boosting, the residual-target GB, and the Neural Network. Even with a *correct* delta the residual is not zero — discrete rebalancing (gamma risk), vol drift (vega risk) and quote rounding contribute — so a lower RMSE is **suggestive** of a better hedge ratio, not definitive proof.

### 7.2 — Headline Result: The Black Formula Is Hard to Beat

The empirical ranking on the held-out test fold (Sec. 7.7) is unambiguous:

1. **IV-Black 76 (analytic)** dominates every other estimator by an order of magnitude: $\text{MAE}_{\text{call}} \approx 5\times 10^{-3}$ vs $\sim 8\times 10^{-2}$ for the best ML model. This is unsurprising: $\sigma^{\text{iv}}$ is back-solved from the same equation that defines $\Delta$, so the IV-Black delta reproduces the vendor delta up to numerical tolerance and quote rounding. It is a useful **practical lower bound** for any model supervised on the quoted delta and not a fair target for ML to beat.

2. **RV-Black 76 (realised vol)** is the *honest* benchmark — the formula a desk would use without buying a vendor IV surface. On our run RV-Black achieves $\text{MAE}_{\text{call}} \approx 0.034$ (calls) and $\sim 0.04$ (puts).

3. **None of the ML/DL models beat RV-Black on raw-target MAE.** Random Forest, Gradient Boosting, Neural Network and IV-augmented GB all post MAE in the 0.08–0.11 range — about 2–3× worse than RV-Black. Linear Ridge/Lasso are worse still.

4. **Hedge-error RMSE tells the same story.** Averaged across strikes, RV-Black is at the low end of the daily tracking RMSE, beaten only by the market-quoted delta itself (the trivial control); the ML/DL deltas track the call P&L noticeably worse than RV-Black.

This is a stronger academic conclusion than the opposite would have been: for the assignment-prescribed feature set $(F, K, \tau, \sigma)$ on a small panel of five strikes, a **structurally correct constant-vol formula** strictly dominates the same 5×5 hyperparameter ML ensemble.

### 7.3 — Why ML Loses on Raw-Target Supervision

The raw-target ML models *try* to learn $\Phi(d_1)$ from $(F/K, \tau, \sigma^{\text{real}})$. Two structural disadvantages explain the 2–3× gap to RV-Black:

1. **Function-class bias.** Trees and shallow MLPs are universal approximators in the limit but allocate complexity locally; the smooth, *exponentially monotonic* $\Phi(d_1)$ is hard to recover with axis-aligned splits or piecewise-linear ReLU regions on $\sim 1700$ training rows over only 5 strikes.
2. **Vol mis-specification is shared.** When fed $\sigma^{\text{real}}$ rather than $\sigma^{\text{iv}}$, the ML model inherits the same constant-vol bias as RV-Black — but on top of that pays an approximation tax for not knowing the formula.

Even feeding back-solved IV directly (Sec. 4.7) does not fix this: the *value* of $\sigma^{\text{iv}}$ is correct, but a tree/NN cannot interpolate the precise non-linear shape of $\Phi(d_1)$ on the available data. The analytic IV-Black formula does that for free. This is exactly why the IV-Black entry in the comparison table is essentially a lower-bound oracle and not a fair target.

### 7.4 — Where ML Genuinely Helps: the Residual / Smile-Correction View

If the ML model is asked instead to predict the **residual** $\Delta^{\text{mkt}} - \Delta^{\text{RV-Black}}$ — i.e. the smile/skew correction layer that a constant-vol formula structurally cannot capture — the comparison becomes meaningful: the residual mean is small, its sign flips with strike (skew signature), and a tree/NN does have a chance to fit it without competing with $\Phi(d_1)$.

In this run the residual-target Gradient Boosting ("GB (residual)" in the tables) is closer to RV-Black than the raw-target GB and produces a measurable, strike-dependent correction. It does not (and cannot) beat IV-Black, since IV-Black already absorbs the smile via $\sigma^{\text{iv}}$, but it does the right thing: it learns *deviations from Black*, not the whole delta surface.

### 7.5 — Similarities and Differences Between ML/DL Delta and Black Delta

**Similarities.**
1. **Sigmoid shape preserved.** All ML/DL models reproduce the familiar S-shaped delta in moneyness (Sec. 6.5, 6.8): deep-OTM $\to 0$, deep-ITM $\to 1$, ATM transition.
2. **Moneyness dominates.** Tree-based importance (Sec. 4.4) and SHAP (Sec. 4.6) agree that $F/K$ is the leading predictor, exactly as the analytical $d_1$ formula prescribes.
3. **Approximate put-call parity.** GB call $-$ GB put on the test fold tracks the $\Delta_c-\Delta_p \approx 1$ futures-style line (Sec. 6.7) without any hard constraint. The Neural Network is noisier but not biased.

**Differences.**
1. **Smile bias of RV-Black is real.** The strike-by-strike table (Sec. 6.6) shows the RV-Black MAE *is* uneven across strikes — largest at the wings, smallest near ATM — the canonical signature of a constant-vol misspecification on an equity skew.
2. **ML cannot reproduce $\Phi(d_1)$ exactly.** This is the central reason raw-target ML loses to RV-Black: an analytic formula encodes a non-linear relationship that a small ensemble cannot reverse-engineer.
3. **Convention sensitivity.** Black 76 has *two* delta conventions on futures (premium-paid vs futures-style). On this dataset put-call parity is decisively futures-style ($\Delta_c-\Delta_p \approx 1$, with mean abs error $\sim 0.02$ vs $\sim 0.04$ for the premium-paid theory). We pick the convention from parity, *not* from RV-Black MAE — the latter is a noisy diagnostic that conflates convention error with the realised-vs-implied vol bias.

### 7.6 — Practical Implications & Recommended Hedge

- **Which delta should we hedge with?** Within this dataset, **IV-Black** is best by a wide margin on both MAE and hedge-RMSE; **RV-Black** is the right fallback when an IV surface is unavailable; **ML/DL on raw delta** is *not* recommended on the prescribed feature set; **ML/DL on the residual** is the right place to keep an empirical correction layer (smile/skew) that Black with realised vol misses.
- **Why the residual framing matters.** Trees/NNs on the residual learn *deviations from Black*, which is what a desk actually wants on top of an analytic engine, not a black-box re-derivation of an option-pricing identity.
- **Limitations of the hedge test.** Daily hedge error includes gamma, vega and rebalancing noise. It is consistent with our headline ordering but should not be over-interpreted as a clean ranking of "true" delta accuracy.
- **Model risk.** ML/DL models are data-dependent and may overfit; on a small five-strike panel the Black formula's structural extrapolation is more robust than any non-parametric fit.
- **Transaction costs.** Any improvement in delta accuracy must be weighed against more frequent rebalancing that a time-varying ML delta might require.

### 7.7 — AI-Contribution Disclosure

> **Human contributions:** the financial interpretations in Sections 7.1–7.6, the choice of the risk-free rate, the convention selection driven by put-call parity, the IV-Black analytical benchmark, the residual-target ML formulation, the date-blocked CV iterator, the chronological train/test split design, the self-financing P&L methodology (error definition, comparators including market and IV-Black), the put-call parity diagnostic, the IV selection-bias diagnostic (Sec. 4.1), the apples-to-apples IV-augmented comparison (Sec. 4.7), the risk-free-rate sensitivity (Sec. 3.2bis), and the overall critical analysis are the authors' own intellectual contribution.
>
> **AI contributions:** the initial boilerplate code structure, the first-draft Keras architecture, some matplotlib scaffolding, and some Markdown formatting were assisted by an AI coding tool (GitHub Copilot / Colab AI). Inline `# AI-assistance note: ...` comments mark the cells where this scaffolding was used. All code was reviewed, understood, modified and validated by the authors. Specifically, the IV extraction procedure, IV-Black benchmark, residual-target ML, date-blocked CV, SHAP diagnostics, 3D surface construction, put-call parity analysis, self-financing replication test, and the Section 7 quantitative narrative were designed and reviewed end-to-end by the team; the AI contribution was limited to syntactic help and first-draft code snippets.

In [ ]:
# == 7.7 Headline numerical results (anchors for Sec. 7) ======================
print("=" * 72)
print("  FINAL MODEL COMPARISON -- CALL Delta Prediction (Test Set)")
print("=" * 72)
display(summary.round(5))

print("\n" + "=" * 72)
print("  FINAL MODEL COMPARISON -- PUT Delta Prediction (Test Set)")
print("=" * 72)
display(summary_put.round(5))

print("\n" + "=" * 72)
print("  Self-Financing Hedge-Error RMSE (index-points / day)")
print("=" * 72)
display(te_rmse)

# --- Numeric anchors used in the Sec 7 narrative ---------------------------
def _safe(name, dct):
    return dct[name]["mae"] if name in dct else float("nan")

mae_black_rv_call = _safe("Black 76 (RV)", results)
mae_black_iv_call = _safe("Black 76 (IV)", results)
mae_gb_call       = _safe("Gradient Boosting", results)
mae_gb_res_call   = _safe("GB (residual)",  results)
mae_nn_call       = _safe("Neural Network", results)
mae_nn_res_call   = _safe("NN (residual)",  results)
mae_iv_aug_call   = _safe("GB + IV",        results)

mae_black_rv_put  = _safe("Black 76 (RV)", put_results)
mae_black_iv_put  = _safe("Black 76 (IV)", put_results)
mae_gb_put        = _safe("Gradient Boosting", put_results)
mae_gb_res_put    = _safe("GB (residual)",  put_results)
mae_nn_put        = _safe("Neural Network", put_results)
mae_nn_res_put    = _safe("NN (residual)",  put_results)

def _imp(x, base):
    return (1 - x / base) * 100 if (base and x == x) else float("nan")

improv_iv_call    = _imp(mae_black_iv_call, mae_black_rv_call)
improv_gb_call    = _imp(mae_gb_call,       mae_black_rv_call)
improv_gb_res     = _imp(mae_gb_res_call,   mae_black_rv_call)
improv_nn_call    = _imp(mae_nn_call,       mae_black_rv_call)
improv_nn_res     = _imp(mae_nn_res_call,   mae_black_rv_call)
improv_iv_aug     = _imp(mae_iv_aug_call,   mae_black_rv_call)

# Strike profile of the RV-Black error
if "Black 76 (RV)" in mae_call_by_K.columns:
    bk_by_K = mae_call_by_K["Black 76 (RV)"].dropna()
    K_worst = int(bk_by_K.idxmax()); K_best = int(bk_by_K.idxmin())
    mae_worst = float(bk_by_K.max());  mae_best = float(bk_by_K.min())
else:
    K_worst = K_best = None
    mae_worst = mae_best = float("nan")

print("\nNumeric anchors used in Section 7:")
print(f"  RV-Black call MAE   : {mae_black_rv_call:.5f}")
print(f"  IV-Black call MAE   : {mae_black_iv_call:.5f}   ({improv_iv_call:+.2f} % vs RV-Black)")
print(f"  GB           call MAE   : {mae_gb_call:.5f}   ({improv_gb_call:+.2f} % vs RV-Black)")
print(f"  GB (residual) call MAE  : {mae_gb_res_call:.5f}   ({improv_gb_res:+.2f} % vs RV-Black)")
print(f"  GB + IV      call MAE   : {mae_iv_aug_call:.5f}   ({improv_iv_aug:+.2f} % vs RV-Black)")
print(f"  NN           call MAE   : {mae_nn_call:.5f}   ({improv_nn_call:+.2f} % vs RV-Black)")
print(f"  NN (residual) call MAE  : {mae_nn_res_call:.5f}   ({improv_nn_res:+.2f} % vs RV-Black)")
if K_worst is not None:
    print(f"  RV-Black strike profile: worst @ K={K_worst} (MAE={mae_worst:.5f}); "
          f"best @ K={K_best} (MAE={mae_best:.5f})")
print(f"  RV-Black put  MAE   : {mae_black_rv_put:.5f}")
print(f"  IV-Black put  MAE   : {mae_black_iv_put:.5f}")
print(f"  GB       put  MAE   : {mae_gb_put:.5f}")
print(f"  GB (res) put  MAE   : {mae_gb_res_put:.5f}")
print(f"  NN       put  MAE   : {mae_nn_put:.5f}")
print(f"  NN (res) put  MAE   : {mae_nn_res_put:.5f}")

# Hedge-error reduction headline (averaged across strikes)
hedge_cols = [c for c in te_rmse.columns if c in
              ('market', 'iv_black', 'rv_black', 'gb', 'gb_res', 'nn')]
if hedge_cols:
    rms = te_rmse[hedge_cols].mean()
    print(f"\nHedge-error RMSE (avg across strikes, lower=better):")
    for c in hedge_cols:
        print(f"     {c:>9s} = {rms[c]:.3f}")
    if 'rv_black' in rms and rms['rv_black']:
        for c in hedge_cols:
            if c == 'rv_black':
                continue
            print(f"     {c} vs rv_black : {(1 - rms[c] / rms['rv_black']) * 100:+.2f} % "
                  f"({'better' if rms[c] < rms['rv_black'] else 'worse'})")
print("=" * 72)